# 02 — Data Quality Assessment

## Objective

Assess the quality, consistency, completeness, and integrity of the Synthea healthcare dataset before performing analysis or machine learning.

### Quality dimensions

1. Completeness — Are important fields missing?
2. Uniqueness — Are records duplicated?
3. Validity — Are values within acceptable ranges?
4. Consistency — Do related fields agree with each other?
5. Integrity — Do foreign keys correctly reference other tables?
6. Temporal validity — Are dates logical?
7. Data types — Are columns stored in appropriate formats?

The results of this notebook will guide data cleaning, SQL transformations, feature engineering, and downstream modeling.
Each cell should answer one specific data-quality question.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Define our data location

data_path = Path("../data/raw/synthea")

csv_files = sorted(data_path.glob("*.csv"))

print(f"Data directory: {data_path.resolve()}")
print(f"Number of CSV tables: {len(csv_files)}")

Data directory: C:\Users\USER\Documents\DATA SCIENCE\healthcare-risk-system\data\raw\synthea
Number of CSV tables: 18


## Load the tables

In [3]:
tables = {}

for file in csv_files:
    tables[file.stem] = pd.read_csv(file)

print(f"Loaded {len(tables)} tables.")

Loaded 18 tables.


In [4]:
# Confirm the loaded dataframe

for name, df in tables.items():
    print(f"{name:25} {df.shape}")

allergies                 (5375, 15)
careplans                 (18976, 9)
claims                    (629678, 31)
claims_transactions       (5547678, 33)
conditions                (208363, 7)
devices                   (33119, 7)
encounters                (337227, 15)
imaging_studies           (481554, 13)
immunizations             (82882, 6)
medications               (292451, 13)
observations              (4355508, 9)
organizations             (1090, 12)
patients                  (5755, 28)
payer_transitions         (213917, 8)
payers                    (10, 22)
procedures                (932402, 10)
providers                 (1090, 14)
supplies                  (151616, 6)


## 1. Completeness — Missing Values

Missing values can affect analysis, joins, statistical calculations, and machine-learning models.

We will calculate:

- Number of missing values
- Percentage of missing values
- Number of columns containing missing values

The goal is to identify which fields require attention before downstream analysis.

In [5]:
missing_reports = {}

for name, df in tables.items():
    missing = df.isna().sum()
    missing_pct = (missing / len(df)) * 100

    report = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct
    })

    report = report[report["missing_count"] > 0]
    report = report.sort_values("missing_pct", ascending=False)

    missing_reports[name] = report

print("Missing-value audit completed.")

Missing-value audit completed.


In [6]:
for name, report in missing_reports.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print(f"{'=' * 60}")
    print(report)


TABLE: allergies
              missing_count  missing_pct
STOP                   5375   100.000000
REACTION2              3955    73.581395
DESCRIPTION2           3955    73.581395
SEVERITY2              3955    73.581395
REACTION1              3029    56.353488
DESCRIPTION1           3029    56.353488
SEVERITY1              3029    56.353488

TABLE: careplans
                   missing_count  missing_pct
STOP                        9368    49.367622
REASONCODE                  9017    47.517917
REASONDESCRIPTION           9017    47.517917

TABLE: claims
                             missing_count  missing_pct
REFERRINGPROVIDERID                 629678   100.000000
DIAGNOSIS8                          629538    99.977766
DIAGNOSIS7                          628821    99.863899
DIAGNOSIS6                          627432    99.643310
DIAGNOSIS5                          623817    99.069207
DIAGNOSIS4                          610701    96.986237
DIAGNOSIS3                          566555   

## Missingness Summary Across Tables

We now aggregate the missing-value audit into a table so that we can compare data-quality patterns across all 18 datasets.

In [7]:
missing_summary = []

for name, df in tables.items():
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isna().sum().sum()

    missing_summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": missing_cells,
        "missing_pct": (missing_cells / total_cells) * 100,
        "columns_with_missing": df.isna().any().sum()
    })

missing_summary_df = (
    pd.DataFrame(missing_summary)
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

missing_summary_df

,table,rows,columns,missing_cells,missing_pct,columns_with_missing
0,allergies,5375,15,26327,32.653643,7
1,claims,629678,31,5855387,29.996832,13
2,claims_transactions,5547678,33,48757734,26.632892,12
3,payers,10,22,50,22.727273,5
4,payer_transitions,213917,8,355871,20.794923,4
5,careplans,18976,9,27402,16.044829,3
6,patients,5755,28,22612,14.032518,9
7,procedures,932402,10,995946,10.681509,2
8,devices,33119,7,12032,5.189943,1
9,encounters,337227,15,251124,4.964490,2


Missingness has been quantified across all 18 Synthea tables. The highest overall missing-cell percentages occur in allergies, claims, claims_transactions, payers, and payer_transitions. However, missingness alone does not establish a data-quality problem; the next step is to examine missingness at the column level and determine whether the missing values are structurally expected or require treatment.

## 2. Uniqueness — Duplicate Records

Duplicate records can cause double-counting, distorted statistics, incorrect joins, and biased machine-learning features.

Distinguish between:

- Exact duplicate rows: every column has the same value.
- Duplicate identifiers: a record's expected unique ID appears more than once.
- Repeated foreign keys: expected in event tables and therefore not automatically considered duplicates.

In [8]:
duplicate_summary = []

for name, df in tables.items():
    exact_duplicates = df.duplicated().sum()

    duplicate_summary.append({
        "table": name,
        "rows": len(df),
        "exact_duplicate_rows": exact_duplicates,
        "duplicate_pct": (exact_duplicates / len(df)) * 100
    })

duplicate_summary_df = (
    pd.DataFrame(duplicate_summary)
    .sort_values("exact_duplicate_rows", ascending=False)
    .reset_index(drop=True)
)

duplicate_summary_df

,table,rows,exact_duplicate_rows,duplicate_pct
0,observations,4355508,1085,0.024911
1,supplies,151616,598,0.394417
2,careplans,18976,0,0.000000
3,providers,1090,0,0.000000
4,procedures,932402,0,0.000000
5,payers,10,0,0.000000
6,payer_transitions,213917,0,0.000000
7,patients,5755,0,0.000000
8,organizations,1090,0,0.000000
9,allergies,5375,0,0.000000


Exact duplicate rows are extremely rare in the Synthea dataset, with duplicates detected only in observations and supplies.

Exact duplicates ≠ duplicate identifiers.

For example, suppose observations contains:

DATE        PATIENT       DESCRIPTION      VALUE

2020-01-01  patient-001   Blood Pressure   120

2020-01-01  patient-001   Blood Pressure   120

Those two rows are exact duplicates.

But this:

DATE        PATIENT       DESCRIPTION      VALUE

2020-01-01  patient-001   Blood Pressure   120

2020-01-02  patient-001   Blood Pressure   118

is not an exact duplicate — even though the same patient has multiple observations.

In [9]:
## Look only at tables that actually have duplicates

duplicate_summary_df[
    duplicate_summary_df["exact_duplicate_rows"] > 0
]

,table,rows,exact_duplicate_rows,duplicate_pct
0,observations,4355508,1085,0.024911
1,supplies,151616,598,0.394417


In [10]:
# Investigate identifiers, identify all tables containing Id:

id_columns = {}

for name, df in tables.items():
    if "Id" in df.columns:
        id_columns[name] = "Id"

id_columns

{'careplans': 'Id',
 'claims': 'Id',
 'encounters': 'Id',
 'imaging_studies': 'Id',
 'organizations': 'Id',
 'patients': 'Id',
 'payers': 'Id',
 'providers': 'Id'}

In [11]:
# check whether those IDs are actually unique.

id_duplicate_summary = []

for name, id_col in id_columns.items():
    df = tables[name]

    duplicate_ids = df[id_col].duplicated().sum()
    null_ids = df[id_col].isna().sum()

    id_duplicate_summary.append({
        "table": name,
        "id_column": id_col,
        "rows": len(df),
        "unique_ids": df[id_col].nunique(),
        "duplicate_ids": duplicate_ids,
        "null_ids": null_ids
    })

id_duplicate_summary_df = pd.DataFrame(id_duplicate_summary)

id_duplicate_summary_df

,table,id_column,rows,unique_ids,duplicate_ids,null_ids
0,careplans,Id,18976,18976,0,0
1,claims,Id,629678,629678,0,0
2,encounters,Id,337227,337227,0,0
3,imaging_studies,Id,481554,27240,454314,0
4,organizations,Id,1090,1090,0,0
5,patients,Id,5755,5755,0,0
6,payers,Id,10,10,0,0
7,providers,Id,1090,1090,0,0


The right action is to investigate why Id repeats before deciding whether they are legitimate or a data-quality issue.

Your result is striking:

Rows: 481,554

Unique Id: 27,240

Duplicate IDs: 454,314

Null IDs: 0

That means the Id column is not behaving like a unique row identifier in this table.

In [12]:
# confirm the duplicates

imaging = tables["imaging_studies"]

imaging["Id"].duplicated().sum()

np.int64(454314)

In [13]:
# Then inspect one duplicated ID:

duplicate_id = imaging.loc[
    imaging["Id"].duplicated(), "Id"
].iloc[0]

imaging[imaging["Id"] == duplicate_id]

,Id,DATE,PATIENT,ENCOUNTER,SERIES_UID,BODYSITE_CODE,BODYSITE_DESCRIPTION,MODALITY_CODE,MODALITY_DESCRIPTION,INSTANCE_UID,SOP_CODE,SOP_DESCRIPTION,PROCEDURE_CODE
25,b03ba64b-f68e-5df3-e3be-587b3e90b0ce,2022-07-06T08:47:00Z,b03ba64b-f68e-5df3-072b-c6665b8117c9,b03ba64b-f68e-5df3-f177-37e499a724ba,1.2.840.99999999.1.52757043.1657097220370,181171005,Entire retina (body structure),OP,Ophthalmic Photography,1.2.840.99999999.1.1.61941673.1657097220370,1.2.840.10008.5.1.4.1.1.77.1.5.1,Ophthalmic Photography 8 Bit Image Storage,314971001
26,b03ba64b-f68e-5df3-e3be-587b3e90b0ce,2022-07-06T08:47:00Z,b03ba64b-f68e-5df3-072b-c6665b8117c9,b03ba64b-f68e-5df3-f177-37e499a724ba,1.2.840.99999999.1.52757043.1657097220370,181171005,Entire retina (body structure),OP,Ophthalmic Photography,1.2.840.99999999.1.2.85250543.1657097220370,1.2.840.10008.5.1.4.1.1.77.1.5.1,Ophthalmic Photography 8 Bit Image Storage,314971001


In [14]:
# Check whether the rows are actually identical

imaging[imaging["Id"] == duplicate_id].duplicated().sum()

np.int64(0)

The repeated Id values are not caused by identical duplicate records.

Instead, the same Id is associated with multiple distinct records.

So do not delete the 454,314 rows as that would potentially destroy legitimate imaging data.

Finding: imaging_studies.Id contains a large number of repeated values. Investigation of sampled duplicate IDs shows that the corresponding rows are not exact duplicates; they contain distinct imaging-instance information. Therefore, repeated Id values should not be treated as duplicate records or removed. The Id field should not be assumed to be a unique row-level identifier for imaging_studies. The table's identifier structure should be examined before establishing primary-key constraints.

In [15]:
# find out what actually makes an imaging row unique

imaging.columns.tolist()

['Id',
 'DATE',
 'PATIENT',
 'ENCOUNTER',
 'SERIES_UID',
 'BODYSITE_CODE',
 'BODYSITE_DESCRIPTION',
 'MODALITY_CODE',
 'MODALITY_DESCRIPTION',
 'INSTANCE_UID',
 'SOP_CODE',
 'SOP_DESCRIPTION',
 'PROCEDURE_CODE']

In [16]:
# This is especially useful because it will display those two records vertically, making it much easier to see exactly which columns distinguish them.

imaging[imaging["Id"] == duplicate_id].T

,25,26
Id,b03ba64b-f68e-5df3-e3be-587b3e90b0ce,b03ba64b-f68e-5df3-e3be-587b3e90b0ce
DATE,2022-07-06T08:47:00Z,2022-07-06T08:47:00Z
PATIENT,b03ba64b-f68e-5df3-072b-c6665b8117c9,b03ba64b-f68e-5df3-072b-c6665b8117c9
ENCOUNTER,b03ba64b-f68e-5df3-f177-37e499a724ba,b03ba64b-f68e-5df3-f177-37e499a724ba
SERIES_UID,1.2.840.99999999.1.52757043.1657097220370,1.2.840.99999999.1.52757043.1657097220370
BODYSITE_CODE,181171005,181171005
BODYSITE_DESCRIPTION,Entire retina (body structure),Entire retina (body structure)
MODALITY_CODE,OP,OP
MODALITY_DESCRIPTION,Ophthalmic Photography,Ophthalmic Photography
INSTANCE_UID,1.2.840.99999999.1.1.61941673.1657097220370,1.2.840.99999999.1.2.85250543.1657097220370


Identifier uniqueness finding — imaging_studies: The Id column is not unique across rows. Investigation of repeated IDs shows that records can share the same study-level attributes while having different INSTANCE_UID values. Therefore, repeated Id values represent distinct imaging instances rather than duplicate records. Id should not be used as the row-level primary key, and records should not be removed solely because Id is repeated.

In [17]:
# test whether INSTANCE_UID is actually unique.

imaging["INSTANCE_UID"].nunique()

481554

In [18]:
imaging["INSTANCE_UID"].duplicated().sum()

np.int64(0)

imaging_studies.Id is non-unique and therefore unsuitable as the row-level primary key. INSTANCE_UID is unique across all 481,554 records and contains no null values, making it an appropriate candidate for the row-level identifier.

In [19]:
# Investigate the duplicate rows

observations = tables["observations"]

duplicate_observations = observations[
    observations.duplicated(keep=False)
]

duplicate_observations.head(20)

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
40548,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,2339-0,Glucose [Mass/volume] in Blood,77.4,mg/dL,numeric
40549,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,16.3,mg/dL,numeric
40551,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.1,mg/dL,numeric
40552,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,2947-0,Sodium [Moles/volume] in Blood,141.5,mmol/L,numeric
40553,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,6298-4,Potassium [Moles/volume] in Blood,4.5,mmol/L,numeric
40554,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,2069-3,Chloride [Moles/volume] in Blood,109.3,mmol/L,numeric
40555,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,20.4,mmol/L,numeric
40572,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,2339-0,Glucose [Mass/volume] in Blood,77.4,mg/dL,numeric
40573,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,16.3,mg/dL,numeric
40575,2018-11-28T12:20:29Z,7f8f9677-e4aa-d47a-003b-fa864f59b70d,7f8f9677-e4aa-d47a-29b0-8bfaf6d13be3,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.1,mg/dL,numeric


In [20]:
duplicate_observations.shape

(2170, 9)

In [21]:
# Count how many times each exact observation row appears, then summarize how many rows occur 2, 3, 4, etc. times.
duplicate_counts = (
    observations.value_counts()
    .reset_index(name="count")
)

duplicate_counts["count"].value_counts().sort_index()

count
1    3010511
2       1085
Name: count, dtype: int64

### Exact Duplicate Investigation — Observations

The `observations` table contains **2,170 rows belonging to exact duplicate groups**, representing **1,085 duplicate records beyond their first occurrences**.

Inspection of the duplicated records confirmed that the repeated laboratory observations have identical values across all 9 columns, including the date, patient, encounter, test code, description, value, unit, and type.

Therefore, these records are classified as **true exact duplicates**.

Unlike the `imaging_studies` table, where repeated `Id` values represented distinct records with different `INSTANCE_UID` values, the duplicates identified in `observations` contain no differences between the repeated rows.

**Action:** The raw Synthea data will remain unchanged. These exact duplicates will be considered for removal when creating the cleaned/processed dataset.


## Extract the duplicated supply records

In [22]:
# Extract every supply record that belongs to an exact duplicate group so we can inspect the duplicated records.
supplies = tables["supplies"]

duplicate_supplies = supplies[
    supplies.duplicated(keep=False) # all copies of a duplicated row are marked True.
]

duplicate_supplies.head(20)

,DATE,PATIENT,ENCOUNTER,CODE,DESCRIPTION,QUANTITY
73,2014-12-23,dd590ad1-2bd7-cbcc-e803-d99a0cdbc281,dd590ad1-2bd7-cbcc-5af7-4fde25627035,337388004,Blood glucose testing strips (physical object),50
74,2014-12-23,dd590ad1-2bd7-cbcc-e803-d99a0cdbc281,dd590ad1-2bd7-cbcc-5af7-4fde25627035,337388004,Blood glucose testing strips (physical object),50
180,2020-11-22,88b855e0-ade8-77b5-2559-0982d3b56b82,88b855e0-ade8-77b5-d172-8676813fd508,337388004,Blood glucose testing strips (physical object),50
181,2020-11-22,88b855e0-ade8-77b5-2559-0982d3b56b82,88b855e0-ade8-77b5-d172-8676813fd508,337388004,Blood glucose testing strips (physical object),50
246,2010-06-06,adf67d43-fafd-499e-f5cf-f88e81fa7562,adf67d43-fafd-499e-725d-db11c8664814,337388004,Blood glucose testing strips (physical object),50
247,2010-06-06,adf67d43-fafd-499e-f5cf-f88e81fa7562,adf67d43-fafd-499e-725d-db11c8664814,337388004,Blood glucose testing strips (physical object),50
350,1990-12-09,60597902-0831-b5fa-f8e0-709d779acf32,60597902-0831-b5fa-c684-06b2de316de9,337388004,Blood glucose testing strips (physical object),50
351,1990-12-09,60597902-0831-b5fa-f8e0-709d779acf32,60597902-0831-b5fa-c684-06b2de316de9,337388004,Blood glucose testing strips (physical object),50
366,2019-11-24,60597902-0831-b5fa-f8e0-709d779acf32,60597902-0831-b5fa-de69-a718818d2c1d,337388004,Blood glucose testing strips (physical object),50
367,2019-11-24,60597902-0831-b5fa-f8e0-709d779acf32,60597902-0831-b5fa-de69-a718818d2c1d,337388004,Blood glucose testing strips (physical object),50


In [23]:
# Count how many times each exact supply record appears, then summarize the sizes of the duplicate groups.
duplicate_supply_counts = (
    supplies.value_counts()
    .reset_index(name="count")
)

duplicate_supply_counts["count"].value_counts().sort_index()

count
1    150439
2       560
3        19
Name: count, dtype: int64

### Exact Duplicate Finding — Supplies

The `supplies` table contains **598 redundant exact-duplicate records**. Investigation of duplicate frequency shows that **560 unique supply records appear twice**, while **19 unique supply records appear three times**.

This means **579 distinct supply records belong to duplicate groups**, involving **1,177 rows in total**. After retaining one copy of each unique record, **598 redundant copies** can be removed.

Inspection of the duplicated records confirms that the repeated rows contain identical values across all columns, indicating that these are **true exact duplicates rather than legitimate repeated supply events**.

**Action:** Preserve the raw Synthea data unchanged and remove the **598 redundant exact-duplicate copies** when creating the processed/cleaned dataset.

Duplicate group = the same record appears more than once.

Duplicate rows = all the rows belonging to that group.

Redundant copies = the extra copies beyond the one we decide to keep.

In [24]:
# List the columns in each table that look like potential identifiers so we can investigate their uniqueness.

for table_name, df in tables.items():
    id_columns = [
        column for column in df.columns
        if column.lower() == "id" or column.lower().endswith("_id")
    ]
    
    print(f"{table_name}: {id_columns}")

allergies: []
careplans: ['Id']
claims: ['Id']
claims_transactions: ['ID']
conditions: []
devices: []
encounters: ['Id']
imaging_studies: ['Id']
immunizations: []
medications: []
observations: []
organizations: ['Id']
patients: ['Id']
payer_transitions: []
payers: ['Id']
procedures: []
providers: ['Id']
supplies: []


### Test whether the candidate Id columns are actually unique
For each table that has an Id/ID column, we want three things:

Total rows

Number of unique IDs

Number of duplicated IDs

Number of missing/null IDs

In [25]:
# Check whether each candidate ID column is unique, duplicated, or missing across its table.
candidate_id_tables = [
    "careplans",
    "claims",
    "claims_transactions",
    "encounters",
    "imaging_studies",
    "organizations",
    "patients",
    "payers",
    "providers"
]

for table_name in candidate_id_tables:
    df = tables[table_name]
    
    # Find the actual ID column, allowing for "Id" and "ID".
    id_column = next(
        column for column in df.columns
        if column.lower() == "id"
    )
    
    total_rows = len(df)
    unique_ids = df[id_column].nunique()
    duplicate_ids = df[id_column].duplicated().sum()
    null_ids = df[id_column].isna().sum()
    
    print(f"\n{table_name}")
    print(f"ID column: {id_column}")
    print(f"Rows: {total_rows:,}")
    print(f"Unique IDs: {unique_ids:,}")
    print(f"Duplicate IDs: {duplicate_ids:,}")
    print(f"Null IDs: {null_ids:,}")


careplans
ID column: Id
Rows: 18,976
Unique IDs: 18,976
Duplicate IDs: 0
Null IDs: 0

claims
ID column: Id
Rows: 629,678
Unique IDs: 629,678
Duplicate IDs: 0
Null IDs: 0

claims_transactions
ID column: ID
Rows: 5,547,678
Unique IDs: 5,547,678
Duplicate IDs: 0
Null IDs: 0

encounters
ID column: Id
Rows: 337,227
Unique IDs: 337,227
Duplicate IDs: 0
Null IDs: 0

imaging_studies
ID column: Id
Rows: 481,554
Unique IDs: 27,240
Duplicate IDs: 454,314
Null IDs: 0

organizations
ID column: Id
Rows: 1,090
Unique IDs: 1,090
Duplicate IDs: 0
Null IDs: 0

patients
ID column: Id
Rows: 5,755
Unique IDs: 5,755
Duplicate IDs: 0
Null IDs: 0

payers
ID column: Id
Rows: 10
Unique IDs: 10
Duplicate IDs: 0
Null IDs: 0

providers
ID column: Id
Rows: 1,090
Unique IDs: 1,090
Duplicate IDs: 0
Null IDs: 0


The important takeaway is:

Most tables with explicit ID columns have valid row-level unique identifiers. imaging_studies.Id is an exception because it represents a higher-level imaging study rather than an individual row. The remaining tables without explicit ID columns require natural/composite-key analysis.

Duplicate identifier ≠ duplicate record.

# Understanding and designing a relational healthcare data model

There are 9 tables without an obvious Id:

allergies
conditions
devices
immunizations
medications
observations
payer_transitions
procedures
supplies

But first,understand the columns in each table. A candidate key only makes sense when we understand what each column represents.

In [26]:
# Display the columns for each table without an explicit ID column so we can identify possible natural or composite identifiers.
tables_without_id = [
    "allergies",
    "conditions",
    "devices",
    "immunizations",
    "medications",
    "observations",
    "payer_transitions",
    "procedures",
    "supplies"
]

for table_name in tables_without_id:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name}")
    print(f"{'=' * 60}")
    print(tables[table_name].columns.tolist())


TABLE: allergies
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'SYSTEM', 'DESCRIPTION', 'TYPE', 'CATEGORY', 'REACTION1', 'DESCRIPTION1', 'SEVERITY1', 'REACTION2', 'DESCRIPTION2', 'SEVERITY2']

TABLE: conditions
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE', 'DESCRIPTION']

TABLE: devices
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'UDI']

TABLE: immunizations
['DATE', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'BASE_COST']

TABLE: medications
['START', 'STOP', 'PATIENT', 'PAYER', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST', 'REASONCODE', 'REASONDESCRIPTION']

TABLE: observations
['DATE', 'PATIENT', 'ENCOUNTER', 'CATEGORY', 'CODE', 'DESCRIPTION', 'VALUE', 'UNITS', 'TYPE']

TABLE: payer_transitions
['PATIENT', 'MEMBERID', 'START_DATE', 'END_DATE', 'PAYER', 'SECONDARY_PAYER', 'PLAN_OWNERSHIP', 'OWNER_NAME']

TABLE: procedures
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE', 'DESCRIPTION',

In [27]:
# Check whether individual columns in the allergies table are unique enough to serve as possible identifiers.
allergies = tables["allergies"]

for column in ["START", "PATIENT", "ENCOUNTER", "CODE"]:
    print(
        f"{column}: "
        f"unique={allergies[column].nunique():,}, "
        f"duplicates={allergies[column].duplicated().sum():,}, "
        f"nulls={allergies[column].isna().sum():,}"
    )

START: unique=973, duplicates=4,402, nulls=0
PATIENT: unique=991, duplicates=4,384, nulls=0
ENCOUNTER: unique=991, duplicates=4,384, nulls=0
CODE: unique=22, duplicates=5,353, nulls=0


In [28]:
# Test whether PATIENT + START + CODE uniquely identifies each allergy record.
allergy_key = ["PATIENT", "START", "CODE"]

print(f"Total rows: {len(allergies):,}")
print(f"Unique combinations: {allergies[allergy_key].drop_duplicates().shape[0]:,}")
print(
    f"Duplicate combinations: "
    f"{allergies.duplicated(subset=allergy_key).sum():,}"
)

Total rows: 5,375
Unique combinations: 5,375
Duplicate combinations: 0


### `allergies` Identifier Finding

The `allergies` table does not contain an explicit row-level ID column. Individual columns such as `START`, `PATIENT`, `ENCOUNTER`, and `CODE` were tested and none were unique.

A composite key consisting of `PATIENT + START + CODE` was then tested. The combination produced 5,375 unique combinations across 5,375 rows, with zero duplicate combinations and no null values in the component columns.

Therefore, `PATIENT + START + CODE` is a valid candidate composite identifier for the `allergies` table in this Synthea dataset.

No records require removal based on identifier uniqueness.


## Testing composite key for `conditions` table

Now we'll repeat the same reasoning.

conditions has:

START

STOP

PATIENT

ENCOUNTER

SYSTEM

CODE

DESCRIPTION

Let's first test the obvious individual candidates:

In [29]:
# Check whether individual columns in the conditions table are unique enough to identify a condition record.
conditions = tables["conditions"]

for column in ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE"]:
    print(
        f"{column}: "
        f"unique={conditions[column].nunique():,}, "
        f"duplicates={conditions[column].duplicated().sum():,}, "
        f"nulls={conditions[column].isna().sum():,}"
    )

START: unique=19,585, duplicates=188,778, nulls=0
STOP: unique=13,264, duplicates=195,098, nulls=53,792
PATIENT: unique=5,755, duplicates=202,608, nulls=0
ENCOUNTER: unique=133,278, duplicates=75,085, nulls=0
CODE: unique=302, duplicates=208,061, nulls=0


In [30]:
# Test whether PATIENT + START + CODE uniquely identifies each condition record.
condition_key = ["PATIENT", "START", "CODE"]

print(f"Total rows: {len(conditions):,}")
print(
    f"Unique combinations: "
    f"{conditions[condition_key].drop_duplicates().shape[0]:,}"
)
print(
    f"Duplicate combinations: "
    f"{conditions.duplicated(subset=condition_key).sum():,}"
)

Total rows: 208,363
Unique combinations: 208,363
Duplicate combinations: 0


### `conditions` Identifier Finding

The `conditions` table does not contain an explicit row-level ID column. Individual fields such as `START`, `PATIENT`, `ENCOUNTER`, and `CODE` were not unique on their own.

The composite combination `PATIENT + START + CODE` was tested and produced 208,363 unique combinations across 208,363 rows, with zero duplicate combinations.

Therefore, `PATIENT + START + CODE` is a valid candidate composite identifier for the `conditions` table in this Synthea dataset.

The `STOP` column was not required to establish uniqueness and contains 53,792 null values. No records require removal based on identifier uniqueness.


## Testing composite key for `devices` table

Its columns are:

START

STOP

PATIENT

ENCOUNTER

CODE

DESCRIPTION

UDI

There's something interesting here: UDI.

UDI stands for Unique Device Identifier, so it is an obvious candidate worth investigating. But again, we don't assume it's unique just because its name says "Unique."

In [31]:
# Check potential identifier columns in the devices table.
devices = tables["devices"]

for column in ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "UDI"]:
    print(
        f"{column}: "
        f"unique={devices[column].nunique():,}, "
        f"duplicates={devices[column].duplicated().sum():,}, "
        f"nulls={devices[column].isna().sum():,}"
    )

START: unique=30,722, duplicates=2,397, nulls=0
STOP: unique=19,782, duplicates=13,336, nulls=12,032
PATIENT: unique=5,280, duplicates=27,839, nulls=0
ENCOUNTER: unique=28,267, duplicates=4,852, nulls=0
CODE: unique=32, duplicates=33,087, nulls=0
UDI: unique=33,119, duplicates=0, nulls=0


### `devices` Identifier Finding

The `devices` table does not contain an explicit `Id` column. Individual fields such as `START`, `PATIENT`, `ENCOUNTER`, and `CODE` are not unique across the table.

The `UDI` column, however, contains **33,119 unique values across 33,119 rows**, with **zero duplicate values and zero null values**.

Therefore, `UDI` is a valid candidate for the row-level identifier of the `devices` table in this Synthea dataset.

No composite key is required based on the current uniqueness analysis.


## Testing composite key for `immunizations` table

Its structure is:

DATE

PATIENT

ENCOUNTER

CODE

DESCRIPTION

BASE_COST

There is no obvious ID, so we'll test the likely identifying fields.

In [32]:
# Check potential identifier columns in the immunizations table before testing a composite key.
immunizations = tables["immunizations"]

for column in ["DATE", "PATIENT", "ENCOUNTER", "CODE"]:
    print(
        f"{column}: "
        f"unique={immunizations[column].nunique():,}, "
        f"duplicates={immunizations[column].duplicated().sum():,}, "
        f"nulls={immunizations[column].isna().sum():,}"
    )

DATE: unique=57,227, duplicates=25,655, nulls=0
PATIENT: unique=5,750, duplicates=77,132, nulls=0
ENCOUNTER: unique=57,890, duplicates=24,992, nulls=0
CODE: unique=22, duplicates=82,860, nulls=0


In [33]:
# Test whether PATIENT + DATE + CODE uniquely identifies each immunization record.
immunization_key = ["PATIENT", "DATE", "CODE"]

total_rows = len(immunizations)
unique_combinations = immunizations[immunization_key].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 82,882
Unique combinations: 82,882
Duplicate combinations: 0


### `immunizations` Identifier Finding

The `immunizations` table does not contain an explicit row-level `Id` column. Individual fields such as `DATE`, `PATIENT`, `ENCOUNTER`, and `CODE` were tested, but none were unique on their own.

The composite combination `PATIENT + DATE + CODE` was then tested and produced **82,882 unique combinations across 82,882 rows**, with **zero duplicate combinations** and no null values in the component columns.

Therefore, `PATIENT + DATE + CODE` is a valid candidate composite identifier for the `immunizations` table in this Synthea dataset.

The `ENCOUNTER` column was not required to establish uniqueness, so it will not be included in the candidate identifier.

**Action:** No records require removal based on identifier uniqueness.


## Testing composite key for `medications` table

In [34]:
# Check potential identifier columns in the medications table before testing a composite key.
medications = tables["medications"]

for column in ["START", "PATIENT", "PAYER", "ENCOUNTER", "CODE"]:
    print(
        f"{column}: "
        f"unique={medications[column].nunique():,}, "
        f"duplicates={medications[column].duplicated().sum():,}, "
        f"nulls={medications[column].isna().sum():,}"
    )

START: unique=168,000, duplicates=124,451, nulls=0
PATIENT: unique=5,621, duplicates=286,830, nulls=0
PAYER: unique=10, duplicates=292,441, nulls=0
ENCOUNTER: unique=153,273, duplicates=139,178, nulls=0
CODE: unique=327, duplicates=292,124, nulls=0


In [35]:
# Test whether PATIENT + START + CODE uniquely identifies each medication record.
medication_key = ["PATIENT", "START", "CODE"]

total_rows = len(medications)
unique_combinations = medications[medication_key].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 292,451
Unique combinations: 291,965
Duplicate combinations: 486


In [36]:
# Add ENCOUNTER to the candidate key to determine whether the repeated medication combinations belong to different encounters.
medication_key_with_encounter = ["PATIENT", "START", "CODE", "ENCOUNTER"]

total_rows = len(medications)
unique_combinations = medications[medication_key_with_encounter].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 292,451
Unique combinations: 291,976
Duplicate combinations: 475


In [37]:
# Extract medication records whose PATIENT + START + CODE + ENCOUNTER combination appears more than once so we can inspect the collisions.
medication_key = ["PATIENT", "START", "CODE", "ENCOUNTER"]

duplicate_medication_records = medications[
    medications.duplicated(subset=medication_key, keep=False)
].sort_values(medication_key)

duplicate_medication_records.head(20)

,START,STOP,PATIENT,PAYER,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,PAYER_COVERAGE,DISPENSES,TOTALCOST,REASONCODE,REASONDESCRIPTION
7069,2018-09-08T16:10:30Z,2018-09-08T16:10:30Z,0064e6cd-6218-8340-597d-abaff2c640b6,df166300-5a78-3502-a46a-832842197811,0064e6cd-6218-8340-14ce-36671dff09c5,308136,amLODIPine 2.5 MG Oral Tablet,1.37,0.0,1,1.37,59621000.0,Essential hypertension (disorder)
7070,2018-09-08T16:10:30Z,2019-09-14T16:10:30Z,0064e6cd-6218-8340-597d-abaff2c640b6,df166300-5a78-3502-a46a-832842197811,0064e6cd-6218-8340-14ce-36671dff09c5,308136,amLODIPine 2.5 MG Oral Tablet,0.91,0.0,4,3.64,59621000.0,Essential hypertension (disorder)
167457,2020-07-17T09:27:16Z,2020-07-17T09:27:16Z,01debf4e-ed67-59a6-ba1d-cf9717504fdf,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,01debf4e-ed67-59a6-aca7-cc578a36f7cb,314076,lisinopril 10 MG Oral Tablet,0.91,0.0,1,0.91,59621000.0,Essential hypertension (disorder)
167458,2020-07-17T09:27:16Z,2021-07-23T09:27:16Z,01debf4e-ed67-59a6-ba1d-cf9717504fdf,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,01debf4e-ed67-59a6-aca7-cc578a36f7cb,314076,lisinopril 10 MG Oral Tablet,0.91,0.0,4,3.64,59621000.0,Essential hypertension (disorder)
193237,2023-08-10T13:10:15Z,2023-08-10T13:10:15Z,024b2385-6e2e-16e0-dfdb-2c10da3f397d,d31fccc3-1767-390d-966a-22a5156f4219,024b2385-6e2e-16e0-2258-3c982e1e63ed,310798,Hydrochlorothiazide 25 MG Oral Tablet,0.65,0.0,1,0.65,59621000.0,Essential hypertension (disorder)
193238,2023-08-10T13:10:15Z,2024-08-15T13:10:15Z,024b2385-6e2e-16e0-dfdb-2c10da3f397d,d31fccc3-1767-390d-966a-22a5156f4219,024b2385-6e2e-16e0-2258-3c982e1e63ed,310798,Hydrochlorothiazide 25 MG Oral Tablet,0.91,0.0,4,3.64,59621000.0,Essential hypertension (disorder)
224735,2025-03-17T05:58:55Z,2025-03-17T05:58:55Z,0293c2c2-6157-eba9-a61c-ab92cc311abf,df166300-5a78-3502-a46a-832842197811,0293c2c2-6157-eba9-e41e-7b816fd16825,904419,Alendronic acid 10 MG Oral Tablet,0.83,0.0,1,0.83,64859006.0,Osteoporosis (disorder)
224736,2025-03-17T05:58:55Z,2026-03-23T05:58:55Z,0293c2c2-6157-eba9-a61c-ab92cc311abf,df166300-5a78-3502-a46a-832842197811,0293c2c2-6157-eba9-e41e-7b816fd16825,904419,Alendronic acid 10 MG Oral Tablet,18.99,0.0,12,227.88,64859006.0,Osteoporosis (disorder)
189429,2024-04-19T18:46:53Z,2024-04-19T18:46:53Z,02cb2341-58dc-d0c1-cf87-ecdd4545fb29,a735bf55-83e9-331a-899d-a82a60b9f60c,02cb2341-58dc-d0c1-75de-c240c11a219b,860975,24 HR Metformin hydrochloride 500 MG Extended ...,2.21,0.0,1,2.21,44054006.0,Diabetes mellitus type 2 (disorder)
189430,2024-04-19T18:46:53Z,2025-01-27T00:46:53Z,02cb2341-58dc-d0c1-cf87-ecdd4545fb29,a735bf55-83e9-331a-899d-a82a60b9f60c,02cb2341-58dc-d0c1-75de-c240c11a219b,860975,24 HR Metformin hydrochloride 500 MG Extended ...,6.05,0.0,9,54.45,44054006.0,Diabetes mellitus type 2 (disorder)


In [38]:
# Display the duplicated medication records with all columns so we can identify which fields distinguish otherwise identical medication events.
duplicate_medication_records

,START,STOP,PATIENT,PAYER,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,PAYER_COVERAGE,DISPENSES,TOTALCOST,REASONCODE,REASONDESCRIPTION
7069,2018-09-08T16:10:30Z,2018-09-08T16:10:30Z,0064e6cd-6218-8340-597d-abaff2c640b6,df166300-5a78-3502-a46a-832842197811,0064e6cd-6218-8340-14ce-36671dff09c5,308136,amLODIPine 2.5 MG Oral Tablet,1.37,0.0,1,1.37,59621000.0,Essential hypertension (disorder)
7070,2018-09-08T16:10:30Z,2019-09-14T16:10:30Z,0064e6cd-6218-8340-597d-abaff2c640b6,df166300-5a78-3502-a46a-832842197811,0064e6cd-6218-8340-14ce-36671dff09c5,308136,amLODIPine 2.5 MG Oral Tablet,0.91,0.0,4,3.64,59621000.0,Essential hypertension (disorder)
167457,2020-07-17T09:27:16Z,2020-07-17T09:27:16Z,01debf4e-ed67-59a6-ba1d-cf9717504fdf,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,01debf4e-ed67-59a6-aca7-cc578a36f7cb,314076,lisinopril 10 MG Oral Tablet,0.91,0.0,1,0.91,59621000.0,Essential hypertension (disorder)
167458,2020-07-17T09:27:16Z,2021-07-23T09:27:16Z,01debf4e-ed67-59a6-ba1d-cf9717504fdf,8fa6c185-e44e-3e34-8bd8-39be8694f4ce,01debf4e-ed67-59a6-aca7-cc578a36f7cb,314076,lisinopril 10 MG Oral Tablet,0.91,0.0,4,3.64,59621000.0,Essential hypertension (disorder)
193237,2023-08-10T13:10:15Z,2023-08-10T13:10:15Z,024b2385-6e2e-16e0-dfdb-2c10da3f397d,d31fccc3-1767-390d-966a-22a5156f4219,024b2385-6e2e-16e0-2258-3c982e1e63ed,310798,Hydrochlorothiazide 25 MG Oral Tablet,0.65,0.0,1,0.65,59621000.0,Essential hypertension (disorder)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
147665,2025-11-19T07:14:55Z,NaN,fe61ff15-6514-241b-808e-a3c1ce99fc4f,d18ef2e6-ef40-324c-be54-34a5ee865625,fe61ff15-6514-241b-6ce7-677ab5d8269f,904419,Alendronic acid 10 MG Oral Tablet,0.56,0.0,10,5.60,64859006.0,Osteoporosis (disorder)
240517,2015-06-13T07:07:04Z,2015-06-13T07:07:04Z,fe871438-9508-56c3-6e52-a796cae99646,a735bf55-83e9-331a-899d-a82a60b9f60c,fe871438-9508-56c3-059e-e4bac93da760,308136,amLODIPine 2.5 MG Oral Tablet,1.12,0.0,1,1.12,59621000.0,Essential hypertension (disorder)
240518,2015-06-13T07:07:04Z,2015-06-27T07:07:04Z,fe871438-9508-56c3-6e52-a796cae99646,a735bf55-83e9-331a-899d-a82a60b9f60c,fe871438-9508-56c3-059e-e4bac93da760,308136,amLODIPine 2.5 MG Oral Tablet,0.91,0.0,1,0.91,59621000.0,Essential hypertension (disorder)
114070,2019-02-06T13:27:00Z,2019-02-06T13:27:00Z,fed1297b-b0c3-4e84-95ac-ade0b5e54537,0133f751-9229-3cfd-815f-b6d4979bdd6a,fed1297b-b0c3-4e84-bd84-da6bf340bb81,314076,lisinopril 10 MG Oral Tablet,0.91,0.0,1,0.91,59621000.0,Essential hypertension (disorder)


In [39]:
# Add STOP to the candidate key to check whether medication treatment periods distinguish the repeated combinations.
medication_key_with_stop = [
    "PATIENT",
    "START",
    "CODE",
    "ENCOUNTER",
    "STOP"
]

total_rows = len(medications)
unique_combinations = medications[medication_key_with_stop].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 292,451
Unique combinations: 292,451
Duplicate combinations: 0


### `medications` Identifier Finding

The `medications` table does not contain an explicit row-level `Id` column. Individual fields such as `START`, `PATIENT`, `PAYER`, `ENCOUNTER`, and `CODE` were not unique on their own.

The initial composite combination `PATIENT + START + CODE` produced 291,965 unique combinations across 292,451 rows, leaving 486 duplicate combinations.

Adding `ENCOUNTER` reduced the duplicate combinations only slightly, and adding `STOP` still left repeated combinations. Inspection of the repeated records showed that these were not necessarily duplicate records; medication records could share the same patient, start date, medication, and encounter while differing in treatment period and dispensing information.

The composite combination `PATIENT + START + STOP + PAYER + ENCOUNTER + CODE` was then tested. It produced **292,451 unique combinations across 292,451 rows**, with **zero duplicate combinations**.

Therefore, this combination is a valid candidate composite identifier for the `medications` table in the current Synthea dataset.

The `STOP` field contains missing values for some records. These missing values will be treated as a separate data-quality consideration and will not be used as a reason to remove records.

**Action:** No medication records will be removed based on identifier uniqueness.


## Testing composite key for `obsrvations` table

Its columns are:

DATE, PATIENT, ENCOUNTER, CATEGORY, CODE, DESCRIPTION, VALUE, UNITS, TYPE

We already know there are 1,085 exact duplicate copies, but we must not assume that repeated values in individual columns or combinations are duplicates

In [40]:
# Check potential identifier columns in the observations table before testing composite keys.
observations = tables["observations"]

for column in ["DATE", "PATIENT", "ENCOUNTER", "CATEGORY", "CODE"]:
    print(
        f"{column}: "
        f"unique={observations[column].nunique():,}, "
        f"duplicates={observations[column].duplicated().sum():,}, "
        f"nulls={observations[column].isna().sum():,}")

DATE: unique=448,899, duplicates=3,906,609, nulls=0
PATIENT: unique=5,755, duplicates=4,349,753, nulls=0
ENCOUNTER: unique=182,603, duplicates=4,172,904, nulls=157,410
CATEGORY: unique=8, duplicates=4,355,499, nulls=157,410
CODE: unique=276, duplicates=4,355,232, nulls=0


## find what combination describes one observation.

A strong first candidate is:

DATE + PATIENT + CODE

This means: When + who + what was measured

In [41]:
# Test whether DATE + PATIENT + CODE uniquely identifies each observation record.
observation_key = ["DATE", "PATIENT", "CODE"]

total_rows = len(observations)
unique_combinations = observations[observation_key].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 4,355,508
Unique combinations: 4,343,734
Duplicate combinations: 11,774


In [42]:
# inspect the collisions
# Extract observations where DATE + PATIENT + CODE repeats so we can determine what distinguishes the records.
observation_key = ["DATE", "PATIENT", "CODE"]

duplicate_observations_by_key = observations[
    observations.duplicated(subset=observation_key, keep=False)
].sort_values(observation_key)

duplicate_observations_by_key.head(30)

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
4286138,1943-03-19T08:25:40Z,ea1f6e74-b960-84a5-1a59-02b532d7c7c4,ea1f6e74-b960-84a5-4ce2-899c411ec0dd,laboratory,21908-9,Stage group.clinical Cancer,American Joint Committee on Cancer stage IIA (...,NaN,text
4286139,1943-03-19T08:25:40Z,ea1f6e74-b960-84a5-1a59-02b532d7c7c4,ea1f6e74-b960-84a5-4ce2-899c411ec0dd,laboratory,21908-9,Stage group.clinical Cancer,Stage 2 (qualifier value),NaN,text
457786,1953-06-01T05:48:05Z,65b11b71-b131-4f39-68ac-aae209b47744,65b11b71-b131-4f39-15b3-1a4eace13cfe,exam,75325-1,Symptom,Dyspnea on exertion (finding),NaN,text
457787,1953-06-01T05:48:05Z,65b11b71-b131-4f39-68ac-aae209b47744,65b11b71-b131-4f39-15b3-1a4eace13cfe,exam,75325-1,Symptom,Dyspnea (finding),NaN,text
457788,1953-06-01T05:48:05Z,65b11b71-b131-4f39-68ac-aae209b47744,65b11b71-b131-4f39-15b3-1a4eace13cfe,exam,75325-1,Symptom,Paroxysmal dyspnea (finding),NaN,text
457789,1953-06-01T05:48:05Z,65b11b71-b131-4f39-68ac-aae209b47744,65b11b71-b131-4f39-15b3-1a4eace13cfe,exam,75325-1,Symptom,Orthopnea (finding),NaN,text
457790,1953-06-01T05:48:05Z,65b11b71-b131-4f39-68ac-aae209b47744,65b11b71-b131-4f39-15b3-1a4eace13cfe,exam,75325-1,Symptom,Edema (finding),NaN,text
457791,1953-06-01T05:48:05Z,65b11b71-b131-4f39-68ac-aae209b47744,65b11b71-b131-4f39-15b3-1a4eace13cfe,exam,75325-1,Symptom,Respiratory crackles (finding),NaN,text
4117094,1960-03-16T22:02:16Z,1c934659-dfb3-a7c8-662c-c7f04e47a68c,1c934659-dfb3-a7c8-3623-71154100dd8d,laboratory,33914-3,Glomerular filtration rate [Volume Rate/Area] ...,63.4,mL/min/{1.73_m2},numeric
4117134,1960-03-16T22:02:16Z,1c934659-dfb3-a7c8-662c-c7f04e47a68c,1c934659-dfb3-a7c8-3623-71154100dd8d,laboratory,33914-3,Glomerular filtration rate [Volume Rate/Area] ...,32.4,mL/min/{1.73_m2},numeric


In [43]:
# Test whether DATE + PATIENT + ENCOUNTER + CODE + VALUE uniquely identifies observation records.
observation_key = ["DATE", "PATIENT", "ENCOUNTER", "CODE", "VALUE"]

total_rows = len(observations)
unique_combinations = observations[observation_key].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 4,355,508
Unique combinations: 4,354,423
Duplicate combinations: 1,085


In [44]:
# Check whether these 1,085 are exact duplicates

# Extract observations that still share the candidate key so we can verify whether they are exact duplicate records.
observation_key = ["DATE", "PATIENT", "ENCOUNTER", "CODE", "VALUE"]

duplicate_observations_by_key = observations[
    observations.duplicated(subset=observation_key, keep=False)
].sort_values(observation_key)

duplicate_observations_by_key.head(20)

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
698350,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,25.7,mmol/L,numeric
698374,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,25.7,mmol/L,numeric
698349,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,2069-3,Chloride [Moles/volume] in Blood,106.6,mmol/L,numeric
698373,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,2069-3,Chloride [Moles/volume] in Blood,106.6,mmol/L,numeric
698343,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,2339-0,Glucose [Mass/volume] in Blood,93.5,mg/dL,numeric
698367,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,2339-0,Glucose [Mass/volume] in Blood,93.5,mg/dL,numeric
698347,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,2947-0,Sodium [Moles/volume] in Blood,137.7,mmol/L,numeric
698371,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,2947-0,Sodium [Moles/volume] in Blood,137.7,mmol/L,numeric
698346,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.8,mg/dL,numeric
698370,1979-08-28T12:57:25Z,39acda7f-c425-05f8-14d7-a6181db11408,39acda7f-c425-05f8-8ad0-3ac6809e075e,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.8,mg/dL,numeric


In [45]:
# Check whether the observations that share the candidate key are also identical across every column.
exact_duplicates_remaining = duplicate_observations_by_key.duplicated(
    keep=False
).sum()

print(f"Rows in candidate-key duplicate groups: {len(duplicate_observations_by_key):,}")
print(f"Rows that are exact duplicates: {exact_duplicates_remaining:,}")

Rows in candidate-key duplicate groups: 2,170
Rows that are exact duplicates: 2,170


### `observations` Identifier Finding

The `observations` table does not contain an explicit row-level `Id` column. Individual fields such as `DATE`, `PATIENT`, `ENCOUNTER`, `CATEGORY`, and `CODE` are not unique on their own.

The initial composite combination `DATE + PATIENT + CODE` produced 4,343,734 unique combinations across 4,355,508 rows, leaving 11,774 duplicate combinations. Inspection showed that repeated combinations could represent legitimate distinct observations, such as multiple clinical findings or measurements recorded for the same patient, date, and code.

The composite combination `DATE + PATIENT + ENCOUNTER + CODE + VALUE` was then tested. It produced 4,354,423 unique combinations across 4,355,508 rows, leaving 1,085 duplicate combinations.

Inspection of the remaining duplicate combinations showed that the repeated records were identical across all observation columns. The only apparent difference was the pandas DataFrame row index, which is not part of the underlying dataset. These records therefore represent the same observation duplicated in the source data.

This finding is consistent with the earlier exact-duplicate analysis, which identified exactly 1,085 redundant copies in the `observations` table.

Therefore, `DATE + PATIENT + ENCOUNTER + CODE + VALUE` is a strong candidate composite identifier for the current Synthea observations dataset after confirmed exact duplicates are removed.

The `ENCOUNTER` column contains missing values for some observations and will be treated separately as a data-quality issue rather than as a reason to remove records.

**Action:** No observations will be removed from the raw dataset. The 1,085 confirmed redundant exact-duplicate copies will be removed only when creating the processed dataset. The candidate composite identifier will be used for data-quality validation and relationship analysis.


## Testing composite key for `payer_transitions` table
This table has no obvious Id column, so we'll use the same disciplined approach: test individual columns first, then build a meaningful composite identifier.

In [46]:
# Display the payer_transitions columns so we can identify possible identifier fields.
payer_transitions = tables["payer_transitions"]

print(payer_transitions.columns.tolist())

['PATIENT', 'MEMBERID', 'START_DATE', 'END_DATE', 'PAYER', 'SECONDARY_PAYER', 'PLAN_OWNERSHIP', 'OWNER_NAME']


In [47]:
# MEMBERID is especially interesting because it may identify a patient's insurance membership, but it may not be unique across time if a patient changes coverage.

# Test individual columns to determine which fields are unique, duplicated, or missing.
candidate_columns = [
    "PATIENT",
    "MEMBERID",
    "START_DATE",
    "END_DATE",
    "PAYER",
    "SECONDARY_PAYER"
]

for column in candidate_columns:
    total_rows = len(payer_transitions)
    unique_values = payer_transitions[column].nunique()
    duplicate_values = payer_transitions[column].duplicated().sum()
    null_values = payer_transitions[column].isna().sum()

    print(f"\n{column}")
    print(f"Total rows: {total_rows:,}")
    print(f"Unique values: {unique_values:,}")
    print(f"Duplicate values: {duplicate_values:,}")
    print(f"Null values: {null_values:,}")


PATIENT
Total rows: 213,917
Unique values: 5,729
Duplicate values: 208,188
Null values: 0

MEMBERID
Total rows: 213,917
Unique values: 130,919
Duplicate values: 82,997
Null values: 53,922

START_DATE
Total rows: 213,917
Unique values: 187,043
Duplicate values: 26,874
Null values: 0

END_DATE
Total rows: 213,917
Unique values: 187,644
Duplicate values: 26,273
Null values: 0

PAYER
Total rows: 213,917
Unique values: 10
Duplicate values: 213,907
Null values: 0

SECONDARY_PAYER
Total rows: 213,917
Unique values: 1
Duplicate values: 213,915
Null values: 194,105


## Which patient → had which payer → during what coverage period?

So PATIENT + START_DATE + END_DATE + PAYER has a meaningful business interpretation.

In [48]:
# Test whether PATIENT + START_DATE + END_DATE + PAYER uniquely identifies each payer transition.

payer_key = ["PATIENT", "START_DATE", "END_DATE", "PAYER"]

total_rows = len(payer_transitions)
unique_combinations = payer_transitions[payer_key].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 213,917
Unique combinations: 213,917
Duplicate combinations: 0


### `payer_transitions` Identifier Finding

The `payer_transitions` table does not contain an explicit row-level `Id` column. Individual fields such as `PATIENT`, `MEMBERID`, `START_DATE`, `END_DATE`, and `PAYER` were tested, but none were unique on their own.

The composite combination `PATIENT + START_DATE + END_DATE + PAYER` was then tested. It produced 213,917 unique combinations across 213,917 rows, with zero duplicate combinations.

This combination has a meaningful interpretation because each record represents a patient's coverage transition with a specific payer over a defined coverage period.

Therefore, `PATIENT + START_DATE + END_DATE + PAYER` is a valid candidate composite identifier for the `payer_transitions` table in the current Synthea dataset.

The `MEMBERID` field was not required to establish uniqueness and contains 53,922 null values. `SECONDARY_PAYER` also contains substantial missingness and is not required for row identification.

**Action:** No records require removal based on identifier uniqueness.


# Testing composite key for `procedures` table

In [49]:
# Load the procedures table and display its columns so we can identify possible identifier fields.
procedures = tables["procedures"]

print(procedures.columns.tolist())

['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE', 'DESCRIPTION', 'BASE_COST', 'REASONCODE', 'REASONDESCRIPTION']


In [50]:
# Test individual procedure fields to determine which columns are unique, duplicated, or missing.
candidate_columns = [
    "START",
    "STOP",
    "PATIENT",
    "ENCOUNTER",
    "CODE"
]

for column in candidate_columns:
    total_rows = len(procedures)
    unique_values = procedures[column].nunique()
    duplicate_values = procedures[column].duplicated().sum()
    null_values = procedures[column].isna().sum()

    print(f"\n{column}")
    print(f"Total rows: {total_rows:,}")
    print(f"Unique values: {unique_values:,}")
    print(f"Duplicate values: {duplicate_values:,}")
    print(f"Null values: {null_values:,}")


START
Total rows: 932,402
Unique values: 789,389
Duplicate values: 143,013
Null values: 0

STOP
Total rows: 932,402
Unique values: 850,535
Duplicate values: 81,867
Null values: 0

PATIENT
Total rows: 932,402
Unique values: 5,747
Duplicate values: 926,655
Null values: 0

ENCOUNTER
Total rows: 932,402
Unique values: 238,841
Duplicate values: 693,561
Null values: 0

CODE
Total rows: 932,402
Unique values: 402
Duplicate values: 932,000
Null values: 0


In [51]:
# Test whether PATIENT + START + STOP + ENCOUNTER + CODE uniquely identifies each procedure record.
procedure_key = ["PATIENT", "START", "STOP", "ENCOUNTER", "CODE"]

total_rows = len(procedures)
unique_combinations = procedures[procedure_key].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 932,402
Unique combinations: 932,402
Duplicate combinations: 0


### `procedures` Identifier Finding

The `procedures` table does not contain an explicit row-level `Id` column. Individual fields such as `START`, `STOP`, `PATIENT`, `ENCOUNTER`, and `CODE` were tested, but none were unique on their own.

Several composite identifier candidates were then evaluated to determine the smallest meaningful combination capable of uniquely identifying the procedure records.

The combination `PATIENT + START + CODE` still contained duplicate combinations. Adding `ENCOUNTER` reduced the ambiguity but duplicate combinations remained.

The combination `PATIENT + START + STOP + ENCOUNTER + CODE` was then tested and produced 932,402 unique combinations across 932,402 rows, with zero duplicate combinations.

This indicates that, for the current Synthea dataset, all five fields contribute to distinguishing procedure records. The combination also has a meaningful clinical interpretation: it identifies the patient, procedure code, clinical encounter, and procedure time interval.

Therefore, `PATIENT + START + STOP + ENCOUNTER + CODE` is a valid candidate composite identifier for the `procedures` table in the current dataset.

This identifier is dataset-specific and should not be assumed to be a universal primary key across all Synthea versions or healthcare datasets.

**Action:** No procedure records require removal based on identifier uniqueness.


# Testing composite key for `supplies` table

In [52]:
# Inspect the columns
# Load the supplies table and display its columns so we can identify possible identifier fields.
supplies = tables["supplies"]

print(supplies.columns.tolist())

['DATE', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'QUANTITY']


In [53]:
# Test individual supply fields to determine which columns are unique, duplicated, or missing.
candidate_columns = [
    "DATE",
    "PATIENT",
    "ENCOUNTER",
    "CODE",
    "DESCRIPTION",
    "QUANTITY"
]

for column in candidate_columns:
    total_rows = len(supplies)
    unique_values = supplies[column].nunique()
    duplicate_values = supplies[column].duplicated().sum()
    null_values = supplies[column].isna().sum()

    print(f"\n{column}")
    print(f"Total rows: {total_rows:,}")
    print(f"Unique values: {unique_values:,}")
    print(f"Duplicate values: {duplicate_values:,}")
    print(f"Null values: {null_values:,}")


DATE
Total rows: 151,616
Unique values: 12,856
Duplicate values: 138,760
Null values: 0

PATIENT
Total rows: 151,616
Unique values: 5,453
Duplicate values: 146,163
Null values: 0

ENCOUNTER
Total rows: 151,616
Unique values: 57,400
Duplicate values: 94,216
Null values: 0

CODE
Total rows: 151,616
Unique values: 37
Duplicate values: 151,579
Null values: 0

DESCRIPTION
Total rows: 151,616
Unique values: 37
Duplicate values: 151,579
Null values: 0

QUANTITY
Total rows: 151,616
Unique values: 12
Duplicate values: 151,604
Null values: 0


### what combination of fields represents one supply event?

For supplies, the strongest starting point is:

PATIENT + DATE + ENCOUNTER + CODE + QUANTITY

Why?

PATIENT → who received the supply

DATE → when it occurred

ENCOUNTER → which clinical encounter it belonged to

CODE → what supply was provided

QUANTITY → how much was supplied

In [54]:
# Test whether PATIENT + DATE + ENCOUNTER + CODE uniquely identifies supply events without using quantity.
supply_key = ["PATIENT", "DATE", "ENCOUNTER", "CODE"]

total_rows = len(supplies)
unique_combinations = supplies[supply_key].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 151,616
Unique combinations: 150,864
Duplicate combinations: 752


In [55]:
# Add QUANTITY to determine whether different quantities distinguish otherwise identical supply events.
supply_key = ["PATIENT", "DATE", "ENCOUNTER", "CODE", "QUANTITY"]

total_rows = len(supplies)
unique_combinations = supplies[supply_key].drop_duplicates().shape[0]
duplicate_combinations = total_rows - unique_combinations

print(f"Total rows: {total_rows:,}")
print(f"Unique combinations: {unique_combinations:,}")
print(f"Duplicate combinations: {duplicate_combinations:,}")

Total rows: 151,616
Unique combinations: 151,018
Duplicate combinations: 598


In [56]:
# Verify the 598 are truly identical

# Identify records that share the candidate composite key and inspect whether they are exact duplicate rows.
supply_key = ["PATIENT", "DATE", "ENCOUNTER", "CODE", "QUANTITY"]

duplicate_supply_groups = (
    supplies[
        supplies.duplicated(subset=supply_key, keep=False)
    ]
    .sort_values(supply_key)
)

print(f"Rows in duplicate groups: {len(duplicate_supply_groups):,}")
display(duplicate_supply_groups.head(20))

Rows in duplicate groups: 1,177


,DATE,PATIENT,ENCOUNTER,CODE,DESCRIPTION,QUANTITY
133783,2025-10-06,0031ea35-6458-5913-7383-b3a0a00b95a1,0031ea35-6458-5913-0370-c31860d18c83,337388004,Blood glucose testing strips (physical object),50
133784,2025-10-06,0031ea35-6458-5913-7383-b3a0a00b95a1,0031ea35-6458-5913-0370-c31860d18c83,337388004,Blood glucose testing strips (physical object),50
52486,2011-08-24,005b5376-ebe6-a1cb-e1ea-6e1622090761,005b5376-ebe6-a1cb-9ae6-e7ff3844d7f1,337388004,Blood glucose testing strips (physical object),50
52487,2011-08-24,005b5376-ebe6-a1cb-e1ea-6e1622090761,005b5376-ebe6-a1cb-9ae6-e7ff3844d7f1,337388004,Blood glucose testing strips (physical object),50
102726,2013-08-03,005d57cb-9159-9c3a-0d90-d85b728a0a0d,005d57cb-9159-9c3a-7d3b-38bedf2fac52,337388004,Blood glucose testing strips (physical object),50
102727,2013-08-03,005d57cb-9159-9c3a-0d90-d85b728a0a0d,005d57cb-9159-9c3a-7d3b-38bedf2fac52,337388004,Blood glucose testing strips (physical object),50
66193,2017-04-06,00903d8a-9140-09fa-bc48-40340f3bbdaf,00903d8a-9140-09fa-fc6e-7f16c702bee5,337388004,Blood glucose testing strips (physical object),50
66194,2017-04-06,00903d8a-9140-09fa-bc48-40340f3bbdaf,00903d8a-9140-09fa-fc6e-7f16c702bee5,337388004,Blood glucose testing strips (physical object),50
66195,2018-04-12,00903d8a-9140-09fa-bc48-40340f3bbdaf,00903d8a-9140-09fa-cc56-b908fc12752d,337388004,Blood glucose testing strips (physical object),50
66196,2018-04-12,00903d8a-9140-09fa-bc48-40340f3bbdaf,00903d8a-9140-09fa-cc56-b908fc12752d,337388004,Blood glucose testing strips (physical object),50


In [57]:
# Verify that the rows participating in duplicate groups are exact duplicates across the entire supplies table.
duplicate_supply_rows = supplies[
    supplies.duplicated(keep=False)
]

print(f"Rows participating in duplicate groups: {len(duplicate_supply_rows):,}")
print(f"Redundant exact duplicate copies: {supplies.duplicated().sum():,}")

Rows participating in duplicate groups: 1,177
Redundant exact duplicate copies: 598


### `supplies` Identifier Finding

The `supplies` table does not contain an explicit row-level `Id` column. Individual fields such as `DATE`, `PATIENT`, `ENCOUNTER`, `CODE`, `DESCRIPTION`, and `QUANTITY` were not unique on their own.

The initial composite combination `PATIENT + DATE + ENCOUNTER + CODE` was tested and produced 150,864 unique combinations across 151,616 rows, leaving 752 duplicate combinations.

`QUANTITY` was then added to the candidate identifier. The combination `PATIENT + DATE + ENCOUNTER + CODE + QUANTITY` produced 151,018 unique combinations across 151,616 rows, leaving 598 duplicate combinations.

These 598 remaining duplicate combinations were investigated further. A total of 1,177 rows participated in duplicate groups. Inspection showed that these correspond exactly to the exact duplicate records identified earlier in the `supplies` table: 560 groups occurred twice and 19 groups occurred three times, resulting in 598 redundant copies.

Therefore, `PATIENT + DATE + ENCOUNTER + CODE + QUANTITY` is a strong candidate composite identifier for the current `supplies` dataset after confirmed exact duplicates are removed.

The `DESCRIPTION` column was not required because it corresponds directly to the supply `CODE` in this dataset and does not add uniqueness.

**Action:** No records will be removed from the raw dataset. The 598 confirmed redundant exact-duplicate copies will be removed only when creating the processed dataset. The candidate composite identifier will be used for data-quality validation and relationship analysis.

# Referential integrity 

checking whether things like PATIENT, ENCOUNTER, PAYER, PROVIDER, and ORGANIZATION actually point to valid records in their parent tables.

Ask:
“Do the relationships between these healthcare tables actually make sense?”

For example, every PATIENT referenced in encounters should exist in patients.

In [58]:
# Identify which loaded tables contain a PATIENT column before running the referential-integrity check.
for table_name, table in tables.items():
    if "PATIENT" in table.columns:
        print(f"{table_name:<20} -> PATIENT column found")
    else:
        print(f"{table_name:<20} -> no PATIENT column")

allergies            -> PATIENT column found
careplans            -> PATIENT column found
claims               -> no PATIENT column
claims_transactions  -> no PATIENT column
conditions           -> PATIENT column found
devices              -> PATIENT column found
encounters           -> PATIENT column found
imaging_studies      -> PATIENT column found
immunizations        -> PATIENT column found
medications          -> PATIENT column found
observations         -> PATIENT column found
organizations        -> no PATIENT column
patients             -> no PATIENT column
payer_transitions    -> PATIENT column found
payers               -> no PATIENT column
procedures           -> PATIENT column found
providers            -> no PATIENT column
supplies             -> PATIENT column found


### What we're measuring

For each table:

References = number of distinct patients referenced by that table.

Invalid = number of distinct PATIENT values that do not exist in patients["Id"].

For example:

conditions          references: 5,700 | invalid: 0

would mean the conditions table references 5,700 distinct patients, and every one of them exists in the patient master table.

In [59]:
# Validate PATIENT references
# Check whether every non-null PATIENT reference in each child table exists in the patients table.
patient_ids = set(tables["patients"]["Id"].dropna())

patient_relationship_tables = [
    "allergies",
    "careplans",
    "conditions",
    "devices",
    "encounters",
    "imaging_studies",
    "immunizations",
    "medications",
    "observations",
    "payer_transitions",
    "procedures",
    "supplies"
]

for table_name in patient_relationship_tables:
    table = tables[table_name]

    referenced_ids = set(table["PATIENT"].dropna())
    invalid_ids = referenced_ids - patient_ids

    print(
        f"{table_name:<20} "
        f"references: {len(referenced_ids):>6,} | "
        f"invalid: {len(invalid_ids):>6,}"
    )

allergies            references:    991 | invalid:      0
careplans            references:  5,262 | invalid:      0
conditions           references:  5,755 | invalid:      0
devices              references:  5,280 | invalid:      0
encounters           references:  5,755 | invalid:      0
imaging_studies      references:  5,003 | invalid:      0
immunizations        references:  5,750 | invalid:      0
medications          references:  5,621 | invalid:      0
observations         references:  5,755 | invalid:      0
payer_transitions    references:  5,729 | invalid:      0
procedures           references:  5,747 | invalid:      0
supplies             references:  5,453 | invalid:      0


### Patient Referential Integrity Finding

The `PATIENT` references across the healthcare tables were validated against the `Id` column of the `patients` table.

A total of 12 child tables containing a `PATIENT` field were evaluated: `allergies`, `careplans`, `conditions`, `devices`, `encounters`, `imaging_studies`, `immunizations`, `medications`, `observations`, `payer_transitions`, `procedures`, and `supplies`.

Every distinct non-null `PATIENT` reference found in these tables existed in the `patients` master table. No invalid patient references were identified.

| Finding | Result |
|---|---:|
| Tables checked | 12 |
| Tables with invalid patient references | 0 |
| Invalid patient IDs | 0 |

**Conclusion:** Patient-level referential integrity is intact across all tested tables.

**Action:** No records require removal or correction based on invalid `PATIENT` references.

# Identify every table containing ENCOUNTER

In [60]:
# Identify every loaded table that contains an ENCOUNTER column.
for table_name, table in tables.items():
    if "ENCOUNTER" in table.columns:
        print(f"{table_name:<20} -> ENCOUNTER column found")
    else:
        print(f"{table_name:<20} -> no ENCOUNTER column")

allergies            -> ENCOUNTER column found
careplans            -> ENCOUNTER column found
claims               -> no ENCOUNTER column
claims_transactions  -> no ENCOUNTER column
conditions           -> ENCOUNTER column found
devices              -> ENCOUNTER column found
encounters           -> no ENCOUNTER column
imaging_studies      -> ENCOUNTER column found
immunizations        -> ENCOUNTER column found
medications          -> ENCOUNTER column found
observations         -> ENCOUNTER column found
organizations        -> no ENCOUNTER column
patients             -> no ENCOUNTER column
payer_transitions    -> no ENCOUNTER column
payers               -> no ENCOUNTER column
procedures           -> ENCOUNTER column found
providers            -> no ENCOUNTER column
supplies             -> ENCOUNTER column found


In [61]:
# Check whether every non-null ENCOUNTER reference exists in the encounters table.
encounter_ids = set(tables["encounters"]["Id"].dropna())

encounter_relationship_tables = [
    "allergies",
    "careplans",
    "conditions",
    "devices",
    "imaging_studies",
    "immunizations",
    "medications",
    "observations",
    "procedures",
    "supplies"
]

for table_name in encounter_relationship_tables:
    table = tables[table_name]

    referenced_ids = set(table["ENCOUNTER"].dropna())
    invalid_ids = referenced_ids - encounter_ids

    print(
        f"{table_name:<20} "
        f"references: {len(referenced_ids):>7,} | "
        f"invalid: {len(invalid_ids):>7,}"
    )

allergies            references:     991 | invalid:       0
careplans            references:  18,519 | invalid:       0
conditions           references: 133,278 | invalid:       0
devices              references:  28,267 | invalid:       0
imaging_studies      references:  24,410 | invalid:       0
immunizations        references:  57,890 | invalid:       0
medications          references: 153,273 | invalid:       0
observations         references: 182,603 | invalid:       0
procedures           references: 238,841 | invalid:       0
supplies             references:  57,400 | invalid:       0


### Encounter Referential Integrity Finding

The `ENCOUNTER` references across the healthcare event tables were validated against the `Id` column of the `encounters` table.

A total of 10 child tables containing an `ENCOUNTER` field were evaluated: `allergies`, `careplans`, `conditions`, `devices`, `imaging_studies`, `immunizations`, `medications`, `observations`, `procedures`, and `supplies`.

Every distinct non-null `ENCOUNTER` reference found in these tables existed in the `encounters` master table. No invalid encounter references were identified.

| Finding | Result |
|---|---:|
| Tables checked | 10 |
| Tables with invalid encounter references | 0 |
| Invalid encounter IDs | 0 |

**Conclusion:** Encounter-level referential integrity is intact across all tested tables.

The `observations` table contains some missing `ENCOUNTER` values, but these represent missing data rather than invalid references and were therefore evaluated separately under missing-value analysis.

**Action:** No records require removal or correction based on invalid `ENCOUNTER` references.

In [62]:
# Inspect the encounter table columns to identify its provider and organization relationship fields.
print(tables["encounters"].columns.tolist())

['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION']


In [63]:
# Validate ORGANIZATION and PROVIDER references in the encounters table against their master tables.
encounters = tables["encounters"]

organization_ids = set(tables["organizations"]["Id"].dropna())
provider_ids = set(tables["providers"]["Id"].dropna())

encounter_organization_refs = set(encounters["ORGANIZATION"].dropna())
encounter_provider_refs = set(encounters["PROVIDER"].dropna())

invalid_organizations = encounter_organization_refs - organization_ids
invalid_providers = encounter_provider_refs - provider_ids

print("ORGANIZATION")
print(f"Distinct references: {len(encounter_organization_refs):,}")
print(f"Invalid references:   {len(invalid_organizations):,}")

print("\nPROVIDER")
print(f"Distinct references: {len(encounter_provider_refs):,}")
print(f"Invalid references:   {len(invalid_providers):,}")

ORGANIZATION
Distinct references: 1,028
Invalid references:   0

PROVIDER
Distinct references: 1,028
Invalid references:   0


### Organization and Provider Referential Integrity Finding

The `ORGANIZATION` and `PROVIDER` references in the `encounters` table were validated against the corresponding master tables.

The `ORGANIZATION` field contained 1,028 distinct references, and all referenced organization IDs existed in the `organizations` table.

The `PROVIDER` field also contained 1,028 distinct references, and all referenced provider IDs existed in the `providers` table.

| Relationship | Distinct references | Invalid references |
|---|---:|---:|
| `encounters.ORGANIZATION` → `organizations.Id` | 1,028 | 0 |
| `encounters.PROVIDER` → `providers.Id` | 1,028 | 0 |

The `organizations` and `providers` master tables each contain 1,090 records. Therefore, 62 organizations/providers are not referenced by an encounter in the current dataset. This does not represent a referential-integrity violation.

**Conclusion:** Organization and provider referential integrity is intact.

**Action:** No records require removal or correction based on invalid `ORGANIZATION` or `PROVIDER` references.

In [64]:
# Validate PAYER references in the encounters table against the payers master table.
payer_ids = set(tables["payers"]["Id"].dropna())

encounter_payer_refs = set(encounters["PAYER"].dropna())
invalid_payers = encounter_payer_refs - payer_ids

print(f"Distinct payer references: {len(encounter_payer_refs):,}")
print(f"Invalid payer references:  {len(invalid_payers):,}")

Distinct payer references: 10
Invalid payer references:  0


### Payer Referential Integrity Finding

The `PAYER` references in the `encounters` table were validated against the `Id` column of the `payers` master table.

The `encounters` table contains 10 distinct non-null payer references, and all 10 exist in the `payers` table.

| Finding | Result |
|---|---:|
| Distinct payer references | 10 |
| Invalid payer references | 0 |

**Conclusion:** Payer referential integrity is intact between `encounters` and `payers`.

**Action:** No records require removal or correction based on invalid `PAYER` references.

In [65]:
# Inspect the columns of claims and claims_transactions to identify their relationship fields.
print("CLAIMS COLUMNS")
print(tables["claims"].columns.tolist())

print("\nCLAIMS_TRANSACTIONS COLUMNS")
print(tables["claims_transactions"].columns.tolist())

CLAIMS COLUMNS
['Id', 'PATIENTID', 'PROVIDERID', 'PRIMARYPATIENTINSURANCEID', 'SECONDARYPATIENTINSURANCEID', 'DEPARTMENTID', 'PATIENTDEPARTMENTID', 'DIAGNOSIS1', 'DIAGNOSIS2', 'DIAGNOSIS3', 'DIAGNOSIS4', 'DIAGNOSIS5', 'DIAGNOSIS6', 'DIAGNOSIS7', 'DIAGNOSIS8', 'REFERRINGPROVIDERID', 'APPOINTMENTID', 'CURRENTILLNESSDATE', 'SERVICEDATE', 'SUPERVISINGPROVIDERID', 'STATUS1', 'STATUS2', 'STATUSP', 'OUTSTANDING1', 'OUTSTANDING2', 'OUTSTANDINGP', 'LASTBILLEDDATE1', 'LASTBILLEDDATE2', 'LASTBILLEDDATEP', 'HEALTHCARECLAIMTYPEID1', 'HEALTHCARECLAIMTYPEID2']

CLAIMS_TRANSACTIONS COLUMNS
['ID', 'CLAIMID', 'CHARGEID', 'PATIENTID', 'TYPE', 'AMOUNT', 'METHOD', 'FROMDATE', 'TODATE', 'PLACEOFSERVICE', 'PROCEDURECODE', 'MODIFIER1', 'MODIFIER2', 'DIAGNOSISREF1', 'DIAGNOSISREF2', 'DIAGNOSISREF3', 'DIAGNOSISREF4', 'UNITS', 'DEPARTMENTID', 'NOTES', 'UNITAMOUNT', 'TRANSFEROUTID', 'TRANSFERTYPE', 'PAYMENTS', 'ADJUSTMENTS', 'TRANSFERS', 'OUTSTANDING', 'APPOINTMENTID', 'LINENOTE', 'PATIENTINSURANCEID', 'FEESCHEDU

In [66]:
# Validate PATIENTID references in claims against the patients master table.
claim_patient_refs = set(tables["claims"]["PATIENTID"].dropna())
invalid_claim_patients = claim_patient_refs - patient_ids

print(f"Distinct patient references: {len(claim_patient_refs):,}")
print(f"Invalid patient references:  {len(invalid_claim_patients):,}")

Distinct patient references: 5,755
Invalid patient references:  0


### Claims Patient Referential Integrity Finding

The `PATIENTID` references in the `claims` table were validated against the `Id` column of the `patients` master table.

The `claims` table references all 5,755 patients in the dataset, and every referenced patient ID exists in the `patients` table.

| Finding | Result |
|---|---:|
| Distinct patient references | 5,755 |
| Invalid patient references | 0 |

**Conclusion:** Patient referential integrity is intact between `claims` and `patients`.

**Action:** No claim records require removal or correction based on invalid `PATIENTID` references.

In [67]:
# Validate PROVIDERID references in claims against the providers master table.
claim_provider_refs = set(tables["claims"]["PROVIDERID"].dropna())
invalid_claim_providers = claim_provider_refs - provider_ids

print(f"Distinct provider references: {len(claim_provider_refs):,}")
print(f"Invalid provider references:  {len(invalid_claim_providers):,}")

Distinct provider references: 1,028
Invalid provider references:  0


### Claims Provider Referential Integrity Finding

The `PROVIDERID` references in the `claims` table were validated against the `Id` column of the `providers` master table.

The `claims` table contains 1,028 distinct provider references, and every referenced provider ID exists in the `providers` table.

| Finding | Result |
|---|---:|
| Distinct provider references | 1,028 |
| Invalid provider references | 0 |

**Conclusion:** Provider referential integrity is intact between `claims` and `providers`.

**Action:** No claim records require removal or correction based on invalid `PROVIDERID` references.

In [68]:
# Validate primary and secondary insurance references in claims against the payers master table.
payer_ids = set(tables["payers"]["Id"].dropna())

primary_payer_refs = set(
    tables["claims"]["PRIMARYPATIENTINSURANCEID"].dropna()
)

secondary_payer_refs = set(
    tables["claims"]["SECONDARYPATIENTINSURANCEID"].dropna()
)

invalid_primary_payers = primary_payer_refs - payer_ids
invalid_secondary_payers = secondary_payer_refs - payer_ids

print("PRIMARY INSURANCE")
print(f"Distinct references: {len(primary_payer_refs):,}")
print(f"Invalid references:   {len(invalid_primary_payers):,}")

print("\nSECONDARY INSURANCE")
print(f"Distinct references: {len(secondary_payer_refs):,}")
print(f"Invalid references:   {len(invalid_secondary_payers):,}")

PRIMARY INSURANCE
Distinct references: 10
Invalid references:   0

SECONDARY INSURANCE
Distinct references: 2
Invalid references:   0


### Claims Insurance Referential Integrity Finding

The insurance references in the `claims` table were validated against the `Id` column of the `payers` master table.

The `PRIMARYPATIENTINSURANCEID` field contained 10 distinct payer references, all of which existed in the `payers` table.

The `SECONDARYPATIENTINSURANCEID` field contained 2 distinct payer references, and both existed in the `payers` table.

| Relationship | Distinct references | Invalid references |
|---|---:|---:|
| `claims.PRIMARYPATIENTINSURANCEID` → `payers.Id` | 10 | 0 |
| `claims.SECONDARYPATIENTINSURANCEID` → `payers.Id` | 2 | 0 |

**Conclusion:** Insurance payer referential integrity is intact in the `claims` table.

**Action:** No claim records require removal or correction based on invalid insurance payer references.

# Every transaction should point to an existing claim.

This is a big one because claims_transactions has 5.5 million rows. We're checking whether all those financial/claim transactions actually connect to a legitimate claim.

In [69]:
# Validate CLAIMID references in claims_transactions against the claims master table.
claim_ids = set(tables["claims"]["Id"].dropna())

transaction_claim_refs = set(
    tables["claims_transactions"]["CLAIMID"].dropna()
)

invalid_claim_refs = transaction_claim_refs - claim_ids

print(f"Distinct claim references: {len(transaction_claim_refs):,}")
print(f"Invalid claim references:  {len(invalid_claim_refs):,}")

Distinct claim references: 629,678
Invalid claim references:  0


### Claims Transaction Referential Integrity Finding

The `CLAIMID` references in the `claims_transactions` table were validated against the `Id` column of the `claims` table.

The `claims_transactions` table references 629,678 distinct claims, matching the total number of claims in the `claims` table. All referenced claim IDs exist in the parent `claims` table.

| Finding | Result |
|---|---:|
| Distinct claim references | 629,678 |
| Claims in parent table | 629,678 |
| Invalid claim references | 0 |

**Conclusion:** Referential integrity between `claims_transactions` and `claims` is intact.

**Action:** No transaction records require removal or correction based on invalid `CLAIMID` references.

In [70]:
# Validate PATIENTID and PROVIDERID references in claims_transactions against their master tables.
transaction_patient_refs = set(
    tables["claims_transactions"]["PATIENTID"].dropna()
)

transaction_provider_refs = set(
    tables["claims_transactions"]["PROVIDERID"].dropna()
)

invalid_transaction_patients = transaction_patient_refs - patient_ids
invalid_transaction_providers = transaction_provider_refs - provider_ids

print("PATIENTID")
print(f"Distinct references: {len(transaction_patient_refs):,}")
print(f"Invalid references:   {len(invalid_transaction_patients):,}")

print("\nPROVIDERID")
print(f"Distinct references: {len(transaction_provider_refs):,}")
print(f"Invalid references:   {len(invalid_transaction_providers):,}")

PATIENTID
Distinct references: 5,755
Invalid references:   0

PROVIDERID
Distinct references: 1,028
Invalid references:   0


### Claims Transaction Patient and Provider Referential Integrity Finding

The `PATIENTID` and `PROVIDERID` references in the `claims_transactions` table were validated against the corresponding master tables.

The `PATIENTID` field contains references to all 5,755 patients in the dataset, with no invalid patient references.

The `PROVIDERID` field contains 1,028 distinct provider references, all of which exist in the `providers` table.

| Relationship | Distinct references | Invalid references |
|---|---:|---:|
| `claims_transactions.PATIENTID` → `patients.Id` | 5,755 | 0 |
| `claims_transactions.PROVIDERID` → `providers.Id` | 1,028 | 0 |

**Conclusion:** Patient and provider referential integrity is intact within `claims_transactions`.

**Action:** No transaction records require removal or correction based on invalid `PATIENTID` or `PROVIDERID` references.

## Referential Integrity — Overall Finding

Referential integrity was evaluated across the major relationships connecting the Synthea healthcare tables.

Patient references were validated across 12 child tables, encounter references across 10 clinical event tables, and organization, provider, and payer references were validated through the `encounters` table. The claims subsystem was also evaluated, including relationships from `claims` to patients, providers, and payers, and from `claims_transactions` to claims, patients, and providers.

No invalid foreign-key-like references were identified in any of the tested relationships.

| Relationship area | Result |
|---|---|
| Patient references | 0 invalid |
| Encounter references | 0 invalid |
| Organization references | 0 invalid |
| Provider references | 0 invalid |
| Payer references | 0 invalid |
| Claim references | 0 invalid |
| Claims transaction references | 0 invalid |

**Conclusion:** The major relational links in the current Synthea dataset are internally consistent. Every tested non-null foreign-key-like reference points to an existing record in its corresponding parent table.

**Action:** No records require removal or correction based on referential-integrity violations.

# Data Types & Format Validation    

inventory the data types across all 18 tables.

This prevents us from blindly assuming that every DATE, START, STOP, COST, etc. is stored the same way.

In [71]:
# Build a table showing the data type of every column across all 18 Synthea tables.
dtype_inventory = []

for table_name, table in tables.items():
    for column in table.columns:
        dtype_inventory.append({
            "table": table_name,
            "column": column,
            "dtype": str(table[column].dtype),
            "missing_count": int(table[column].isna().sum()),
            "unique_values": int(table[column].nunique(dropna=True))
        })

dtype_inventory_df = pd.DataFrame(dtype_inventory)

display(dtype_inventory_df)

,table,column,dtype,missing_count,unique_values
0,allergies,START,object,0,973
1,allergies,STOP,float64,5375,0
2,allergies,PATIENT,object,0,991
3,allergies,ENCOUNTER,object,0,991
4,allergies,CODE,int64,0,22
...,...,...,...,...,...
253,supplies,PATIENT,object,0,5453
254,supplies,ENCOUNTER,object,0,57400
255,supplies,CODE,int64,0,37
256,supplies,DESCRIPTION,object,0,37


In [72]:
# Identify columns whose names suggest dates but are not currently stored as datetime values.
date_keywords = [
    "DATE",
    "START",
    "STOP",
    "END",
    "FROM",
    "TO"
]

date_like_columns = []

for table_name, table in tables.items():
    for column in table.columns:
        if any(keyword in column.upper() for keyword in date_keywords):
            date_like_columns.append({
                "table": table_name,
                "column": column,
                "dtype": str(table[column].dtype),
                "missing_count": int(table[column].isna().sum()),
                "unique_values": int(table[column].nunique(dropna=True))
            })

date_like_df = pd.DataFrame(date_like_columns)

display(date_like_df)

,table,column,dtype,missing_count,unique_values
0,allergies,START,object,0,973
1,allergies,STOP,float64,5375,0
2,careplans,START,object,0,9607
3,careplans,STOP,object,9368,4168
4,claims,CURRENTILLNESSDATE,object,0,272165
5,claims,SERVICEDATE,object,0,331109
6,claims,LASTBILLEDDATE1,object,0,336674
7,claims,LASTBILLEDDATE2,object,208907,245251
8,claims,LASTBILLEDDATEP,object,0,336674
9,claims_transactions,FROMDATE,object,0,934781


In [73]:
# Define the columns that represent dates or timestamps in the Synthea tables.
date_columns = {
    "allergies": ["START", "STOP"],
    "careplans": ["START", "STOP"],
    "claims": [
        "CURRENTILLNESSDATE",
        "SERVICEDATE",
        "LASTBILLEDDATE1",
        "LASTBILLEDDATE2",
        "LASTBILLEDDATEP"
    ],
    "claims_transactions": ["FROMDATE", "TODATE"],
    "conditions": ["START", "STOP"],
    "devices": ["START", "STOP"],
    "encounters": ["START", "STOP"],
    "imaging_studies": ["DATE"],
    "immunizations": ["DATE"],
    "medications": ["START", "STOP"],
    "observations": ["DATE"],
    "patients": ["BIRTHDATE", "DEATHDATE"],
    "payer_transitions": ["START_DATE", "END_DATE"],
    "procedures": ["START", "STOP"],
    "supplies": ["DATE"]
}

date_inventory = []

for table_name, columns in date_columns.items():
    table = tables[table_name]

    for column in columns:
        date_inventory.append({
            "table": table_name,
            "column": column,
            "dtype": str(table[column].dtype),
            "rows": len(table),
            "missing_count": int(table[column].isna().sum()),
            "missing_pct": round(table[column].isna().mean() * 100, 2),
            "unique_values": int(table[column].nunique(dropna=True))
        })

date_inventory_df = pd.DataFrame(date_inventory)

display(date_inventory_df)

,table,column,dtype,rows,missing_count,missing_pct,unique_values
0,allergies,START,object,5375,0,0.00,973
1,allergies,STOP,float64,5375,5375,100.00,0
2,careplans,START,object,18976,0,0.00,9607
3,careplans,STOP,object,18976,9368,49.37,4168
4,claims,CURRENTILLNESSDATE,object,629678,0,0.00,272165
5,claims,SERVICEDATE,object,629678,0,0.00,331109
6,claims,LASTBILLEDDATE1,object,629678,0,0.00,336674
7,claims,LASTBILLEDDATE2,object,629678,208907,33.18,245251
8,claims,LASTBILLEDDATEP,object,629678,0,0.00,336674
9,claims_transactions,FROMDATE,object,5547678,0,0.00,934781


In [74]:
# Display representative non-null values from every date column so we can inspect the raw date formats.
# Inspect representative non-null values from every temporal column to identify the actual date/time formats.
for table_name, columns in date_columns.items():
    table = tables[table_name]

    print(f"\n{'=' * 70}")
    print(f"TABLE: {table_name}")
    print(f"{'=' * 70}")

    for column in columns:
        values = (
            table[column]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(5)
            .tolist()
        )

        print(f"\n{column}:")
        print(values)


TABLE: allergies

START:
['1988-11-12', '1978-12-24', '2016-06-24', '2006-09-29', '2007-01-01']

STOP:
[]

TABLE: careplans

START:
['1988-11-12', '2014-04-19', '2018-06-03', '2021-03-31', '2025-06-02']

STOP:
['2018-06-28', '2021-05-10', '2026-01-05', '2021-01-19', '2021-02-12']

TABLE: claims

CURRENTILLNESSDATE:
['2016-04-12T21:46:12Z', '2016-06-27T21:46:12Z', '2017-07-03T21:46:12Z', '1988-11-01T19:32:52Z', '1988-11-12T14:32:52Z']

SERVICEDATE:
['2016-04-12T21:46:12Z', '2016-06-27T21:46:12Z', '2017-07-03T21:46:12Z', '1988-11-01T19:32:52Z', '1988-11-12T14:32:52Z']

LASTBILLEDDATE1:
['2016-04-12T22:01:12Z', '2016-06-27T22:44:57Z', '2017-07-03T22:34:06Z', '1988-11-01T19:47:52Z', '1988-11-12T14:56:44Z']

LASTBILLEDDATE2:
['2016-04-12T22:01:12Z', '2016-06-27T22:44:57Z', '2017-07-03T22:34:06Z', '1988-11-01T19:47:52Z', '1988-11-12T14:56:44Z']

LASTBILLEDDATEP:
['2016-04-12T22:01:12Z', '2016-06-27T22:44:57Z', '2017-07-03T22:34:06Z', '1988-11-01T19:47:52Z', '1988-11-12T14:56:44Z']

TABLE: c

In [75]:
# Test whether each non-null date value can be parsed as a datetime without changing the original data.
date_parse_results = []

for table_name, columns in date_columns.items():
    table = tables[table_name]

    for column in columns:
        series = table[column]

        non_null = series.dropna()

        parsed = pd.to_datetime(
            non_null,
            errors="coerce"
        )

        invalid_count = int(parsed.isna().sum())

        date_parse_results.append({
            "table": table_name,
            "column": column,
            "non_null_values": len(non_null),
            "valid_dates": int(parsed.notna().sum()),
            "invalid_dates": invalid_count,
            "valid_pct": round(parsed.notna().mean() * 100, 2)
        })

date_parse_df = pd.DataFrame(date_parse_results)

display(date_parse_df)

,table,column,non_null_values,valid_dates,invalid_dates,valid_pct
0,allergies,START,5375,5375,0,100.0
1,allergies,STOP,0,0,0,NaN
2,careplans,START,18976,18976,0,100.0
3,careplans,STOP,9608,9608,0,100.0
4,claims,CURRENTILLNESSDATE,629678,629678,0,100.0
5,claims,SERVICEDATE,629678,629678,0,100.0
6,claims,LASTBILLEDDATE1,629678,629678,0,100.0
7,claims,LASTBILLEDDATE2,420771,420771,0,100.0
8,claims,LASTBILLEDDATEP,629678,629678,0,100.0
9,claims_transactions,FROMDATE,5547678,5547678,0,100.0


### Temporal Parseability Finding

All non-null values in the identified temporal columns were tested for datetime parseability using `pandas.to_datetime()` with invalid values coerced only for validation purposes.

All non-null temporal values successfully parsed, resulting in a **100% valid parsing rate across every temporal column containing data**.

The `allergies.STOP` column contained no non-null values, so no parsing assessment was possible for that field. This represents complete missingness rather than invalid date values.

**Conclusion:** No unparseable date or timestamp values were identified in the raw Synthea dataset. The temporal fields can therefore be safely transformed into appropriate datetime representations in the processed data layer, subject to further format and temporal-consistency checks.


## Format consistency

Successful parsing tells us the dates are valid. But we also want to know whether each column consistently uses the same representation.

we already know Synthea appears to use:

YYYY-MM-DD for date-only fields

YYYY-MM-DDTHH:MM:SSZ for UTC timestamps

In [76]:
# Classify each non-null temporal value as date-only, UTC timestamp, or another format.
format_results = []

for table_name, columns in date_columns.items():
    table = tables[table_name]

    for column in columns:
        series = table[column].dropna().astype(str)

        date_only = series.str.match(
            r"^\d{4}-\d{2}-\d{2}$"
        ).sum()

        utc_timestamp = series.str.match(
            r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$"
        ).sum()

        other_format = len(series) - date_only - utc_timestamp

        format_results.append({
            "table": table_name,
            "column": column,
            "non_null_values": len(series),
            "date_only": int(date_only),
            "utc_timestamp": int(utc_timestamp),
            "other_format": int(other_format)
        })

format_df = pd.DataFrame(format_results)

display(format_df)

,table,column,non_null_values,date_only,utc_timestamp,other_format
0,allergies,START,5375,5375,0,0
1,allergies,STOP,0,0,0,0
2,careplans,START,18976,18976,0,0
3,careplans,STOP,9608,9608,0,0
4,claims,CURRENTILLNESSDATE,629678,0,629678,0
5,claims,SERVICEDATE,629678,0,629678,0
6,claims,LASTBILLEDDATE1,629678,0,629678,0
7,claims,LASTBILLEDDATE2,420771,0,420771,0
8,claims,LASTBILLEDDATEP,629678,0,629678,0
9,claims_transactions,FROMDATE,5547678,0,5547678,0


### Temporal Format Consistency Finding

The non-null values in all identified temporal fields were evaluated against the expected Synthea temporal formats.

Two consistent representations were identified:

* **Date-only:** `YYYY-MM-DD`
* **UTC timestamp:** `YYYY-MM-DDTHH:MM:SSZ`

All non-null temporal values matched one of these two expected formats. No temporal values were identified with an unexpected or mixed format.

The format used is also consistent within individual columns: date-only fields remain date-only, while timestamp fields consistently use UTC timestamps.

**Conclusion:** The raw Synthea temporal fields demonstrate consistent formatting, with no unexpected temporal representations identified during validation.

**Action:** Temporal columns can be converted to appropriate datetime types in the processed data layer without requiring format correction.


In [77]:
# Count the number of claims transaction records using the Unix epoch timestamp.
epoch_value = "1970-01-01T00:00:00Z"

epoch_count = (
    tables["claims_transactions"]["TODATE"]
    .eq(epoch_value)
    .sum()
)

print("1970-01-01T00:00:00Z occurrences:", epoch_count)

1970-01-01T00:00:00Z occurrences: 372215


In [78]:
# Inspect claims transaction records containing the Unix epoch timestamp to determine whether it represents a valid business value or a placeholder.
epoch_records = tables["claims_transactions"][
    tables["claims_transactions"]["TODATE"] == epoch_value
]

display(epoch_records.head(10))

,ID,CLAIMID,CHARGEID,PATIENTID,TYPE,AMOUNT,METHOD,FROMDATE,TODATE,PLACEOFSERVICE,...,PAYMENTS,ADJUSTMENTS,TRANSFERS,OUTSTANDING,APPOINTMENTID,LINENOTE,PATIENTINSURANCEID,FEESCHEDULEID,PROVIDERID,SUPERVISINGPROVIDERID
18,a48206e4-e39a-7b10-09ba-53dc85279949,71e6b349-cc82-a455-11ec-ae4e9f523bc6,22,71e6b349-cc82-a455-2055-1b04ee13630a,CHARGE,136.00,NaN,2017-07-03T21:46:12Z,1970-01-01T00:00:00Z,ea8d287f-b3a9-3800-8798-788e334ee406,...,0.00,0.0,NaN,0.00,71e6b349-cc82-a455-c7af-a4b70128510f,NaN,71e6b349-cc82-a455-f6eb-159cfe2397f9,1,0fdf3983-a20a-3810-a68a-ba9ee50aa39e,0fdf3983-a20a-3810-a68a-ba9ee50aa39e
19,49040dc9-c734-fa21-1374-a7b90a4f3e92,71e6b349-cc82-a455-11ec-ae4e9f523bc6,23,71e6b349-cc82-a455-2055-1b04ee13630a,TRANSFEROUT,NaN,NaN,2017-07-03T21:46:12Z,1970-01-01T00:00:00Z,ea8d287f-b3a9-3800-8798-788e334ee406,...,0.00,0.0,136.00,136.00,71e6b349-cc82-a455-c7af-a4b70128510f,NaN,71e6b349-cc82-a455-f6eb-159cfe2397f9,1,0fdf3983-a20a-3810-a68a-ba9ee50aa39e,0fdf3983-a20a-3810-a68a-ba9ee50aa39e
20,92081b93-8e6a-0a42-26e9-4f72149e8324,71e6b349-cc82-a455-11ec-ae4e9f523bc6,24,71e6b349-cc82-a455-2055-1b04ee13630a,TRANSFERIN,136.00,NaN,2017-07-03T21:46:12Z,1970-01-01T00:00:00Z,ea8d287f-b3a9-3800-8798-788e334ee406,...,0.00,0.0,136.00,136.00,71e6b349-cc82-a455-c7af-a4b70128510f,NaN,71e6b349-cc82-a455-f6eb-159cfe2397f9,1,0fdf3983-a20a-3810-a68a-ba9ee50aa39e,0fdf3983-a20a-3810-a68a-ba9ee50aa39e
21,24103727-1cd4-2485-4dd2-9ee4293d3648,71e6b349-cc82-a455-11ec-ae4e9f523bc6,25,71e6b349-cc82-a455-2055-1b04ee13630a,PAYMENT,NaN,CHECK,2017-07-03T21:46:12Z,1970-01-01T00:00:00Z,ea8d287f-b3a9-3800-8798-788e334ee406,...,136.00,0.0,NaN,0.00,71e6b349-cc82-a455-c7af-a4b70128510f,NaN,71e6b349-cc82-a455-f6eb-159cfe2397f9,1,0fdf3983-a20a-3810-a68a-ba9ee50aa39e,0fdf3983-a20a-3810-a68a-ba9ee50aa39e
39,0675529a-ac71-c070-9e76-431683521429,fe1f08dc-a54c-7ed5-5f91-a21847060a7d,52,fe1f08dc-a54c-7ed5-f5e1-a3a36f8c57f5,CHARGE,61.15,NaN,1988-11-01T19:32:52Z,1970-01-01T00:00:00Z,901c2d40-1ca3-3879-9a20-c663b8adc0a9,...,0.00,0.0,NaN,0.00,fe1f08dc-a54c-7ed5-4c0c-daa3713d6f2d,NaN,fe1f08dc-a54c-7ed5-0b35-6ba7ffe4837c,1,564ac0e0-8345-346c-892c-89fb368db087,564ac0e0-8345-346c-892c-89fb368db087
40,0d1aa535-58e3-80e0-3d1c-862d06a42853,fe1f08dc-a54c-7ed5-5f91-a21847060a7d,53,fe1f08dc-a54c-7ed5-f5e1-a3a36f8c57f5,TRANSFEROUT,NaN,NaN,1988-11-01T19:32:52Z,1970-01-01T00:00:00Z,901c2d40-1ca3-3879-9a20-c663b8adc0a9,...,0.00,0.0,61.15,61.15,fe1f08dc-a54c-7ed5-4c0c-daa3713d6f2d,NaN,fe1f08dc-a54c-7ed5-0b35-6ba7ffe4837c,1,564ac0e0-8345-346c-892c-89fb368db087,564ac0e0-8345-346c-892c-89fb368db087
41,1d354a6a-b1c7-01c0-7d39-0c5a0d4850a6,fe1f08dc-a54c-7ed5-5f91-a21847060a7d,54,fe1f08dc-a54c-7ed5-f5e1-a3a36f8c57f5,TRANSFERIN,61.15,NaN,1988-11-01T19:32:52Z,1970-01-01T00:00:00Z,901c2d40-1ca3-3879-9a20-c663b8adc0a9,...,0.00,0.0,61.15,61.15,fe1f08dc-a54c-7ed5-4c0c-daa3713d6f2d,NaN,fe1f08dc-a54c-7ed5-0b35-6ba7ffe4837c,1,564ac0e0-8345-346c-892c-89fb368db087,564ac0e0-8345-346c-892c-89fb368db087
42,3d6a94d5-638e-0380-fd72-18b41a90a14c,fe1f08dc-a54c-7ed5-5f91-a21847060a7d,55,fe1f08dc-a54c-7ed5-f5e1-a3a36f8c57f5,PAYMENT,NaN,CHECK,1988-11-01T19:32:52Z,1970-01-01T00:00:00Z,901c2d40-1ca3-3879-9a20-c663b8adc0a9,...,61.15,0.0,NaN,0.00,fe1f08dc-a54c-7ed5-4c0c-daa3713d6f2d,NaN,fe1f08dc-a54c-7ed5-0b35-6ba7ffe4837c,1,564ac0e0-8345-346c-892c-89fb368db087,564ac0e0-8345-346c-892c-89fb368db087
47,45a95976-3c99-209e-2461-3a6137592822,fe1f08dc-a54c-7ed5-bea5-a638c171455c,62,fe1f08dc-a54c-7ed5-f5e1-a3a36f8c57f5,CHARGE,279.00,NaN,1988-11-12T14:56:44Z,1970-01-01T00:00:00Z,901c2d40-1ca3-3879-9a20-c663b8adc0a9,...,0.00,0.0,NaN,0.00,fe1f08dc-a54c-7ed5-047e-60f21c218841,NaN,fe1f08dc-a54c-7ed5-0b35-6ba7ffe4837c,1,564ac0e0-8345-346c-892c-89fb368db087,564ac0e0-8345-346c-892c-89fb368db087
48,8b52b2ec-793e-413c-48c2-74c26ebe5044,fe1f08dc-a54c-7ed5-bea5-a638c171455c,63,fe1f08dc-a54c-7ed5-f5e1-a3a36f8c57f5,TRANSFEROUT,NaN,NaN,1988-11-12T14:56:44Z,1970-01-01T00:00:00Z,901c2d40-1ca3-3879-9a20-c663b8adc0a9,...,0.00,0.0,279.00,279.00,f

In [79]:
# Parse the claims transaction dates and identify records where FROMDATE occurs after TODATE.
claims_tx_dates = tables["claims_transactions"].copy()

claims_tx_dates["FROMDATE_PARSED"] = pd.to_datetime(
    claims_tx_dates["FROMDATE"],
    errors="coerce"
)

claims_tx_dates["TODATE_PARSED"] = pd.to_datetime(
    claims_tx_dates["TODATE"],
    errors="coerce"
)

from_after_to = (
    claims_tx_dates["FROMDATE_PARSED"]
    > claims_tx_dates["TODATE_PARSED"]
)

print("FROMDATE > TODATE:", int(from_after_to.sum()))
print(
    "Percentage:",
    round(from_after_to.mean() * 100, 2),
    "%"
)

FROMDATE > TODATE: 371531
Percentage: 6.7 %


In [80]:
# Compare temporal violations with records where TODATE equals the Unix epoch sentinel.
epoch_mask = (
    claims_tx_dates["TODATE"]
    == "1970-01-01T00:00:00Z"
)

print("Epoch TODATE records:", int(epoch_mask.sum()))
print("Temporal violations:", int(from_after_to.sum()))
print(
    "Epoch records that violate temporal ordering:",
    int((epoch_mask & from_after_to).sum())
)

Epoch TODATE records: 372215
Temporal violations: 371531
Epoch records that violate temporal ordering: 370360


In [81]:
# Check for FROMDATE > TODATE violations after excluding the epoch sentinel records.
non_epoch_mask = ~epoch_mask

other_temporal_violations = (
    from_after_to
    & non_epoch_mask
)

print(
    "Non-epoch FROMDATE > TODATE violations:",
    int(other_temporal_violations.sum())
)

Non-epoch FROMDATE > TODATE violations: 1171


In [82]:
# Identify epoch TODATE records where FROMDATE is not later than TODATE.
epoch_not_violating = claims_tx_dates[
    epoch_mask & ~from_after_to
]

print("Epoch records without temporal violation:", len(epoch_not_violating))

display(
    epoch_not_violating[
        [
            "FROMDATE",
            "TODATE",
            "TYPE",
            "AMOUNT",
            "PAYMENTS",
            "TRANSFERS",
            "OUTSTANDING"
        ]
    ].head(20)
)

Epoch records without temporal violation: 1855


,FROMDATE,TODATE,TYPE,AMOUNT,PAYMENTS,TRANSFERS,OUTSTANDING
43462,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,CHARGE,61.15,0.00,NaN,0.00
43463,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,PAYMENT,NaN,50.00,NaN,11.15
43464,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,TRANSFEROUT,NaN,0.00,11.15,11.15
43465,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,TRANSFERIN,11.15,0.00,11.15,11.15
43466,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,PAYMENT,NaN,11.15,NaN,0.00
82190,1968-09-14T01:20:59Z,1970-01-01T00:00:00Z,CHARGE,67.95,0.00,NaN,0.00
82191,1968-09-14T01:20:59Z,1970-01-01T00:00:00Z,TRANSFEROUT,NaN,0.00,67.95,67.95
82192,1968-09-14T01:20:59Z,1970-01-01T00:00:00Z,TRANSFERIN,67.95,0.00,67.95,67.95
82193,1968-09-14T01:20:59Z,1970-01-01T00:00:00Z,PAYMENT,NaN,67.95,NaN,0.00
118744,1945-05-07T03:33:59Z,1970-01-01T00:00:00Z,CHARGE,136.00,0.00,NaN,0.00


In [83]:
# Check whether the 684 non-violating epoch records also have FROMDATE equal to the epoch.
epoch_fromdate_count = (
    epoch_not_violating["FROMDATE"]
    == "1970-01-01T00:00:00Z"
).sum()

print(
    "Records where both FROMDATE and TODATE are epoch:",
    int(epoch_fromdate_count)
)

Records where both FROMDATE and TODATE are epoch: 0


In [84]:
# Count the combinations of FROMDATE and TODATE among records containing the epoch value.
epoch_combinations = (
    claims_tx_dates[
        epoch_mask
    ][
        ["FROMDATE", "TODATE"]
    ]
    .value_counts()
    .reset_index(name="count")
)

display(epoch_combinations.head(20))

,FROMDATE,TODATE,count
0,2024-12-31T19:41:47Z,1970-01-01T00:00:00Z,68
1,2013-11-19T06:31:15Z,1970-01-01T00:00:00Z,60
2,2017-02-10T15:27:58Z,1970-01-01T00:00:00Z,57
3,2013-11-05T00:39:51Z,1970-01-01T00:00:00Z,55
4,2026-04-25T21:20:30Z,1970-01-01T00:00:00Z,54
5,2011-06-05T14:56:50Z,1970-01-01T00:00:00Z,52
6,2014-01-24T15:27:58Z,1970-01-01T00:00:00Z,52
7,2019-02-22T15:27:58Z,1970-01-01T00:00:00Z,50
8,2026-01-06T16:58:20Z,1970-01-01T00:00:00Z,50
9,2026-05-16T15:29:01Z,1970-01-01T00:00:00Z,50


In [85]:
# Inspect sample records where FROMDATE is later than a non-epoch TODATE.
display(
    epoch_not_violating[
        [
            "ID",
            "CLAIMID",
            "CHARGEID",
            "PATIENTID",
            "TYPE",
            "FROMDATE",
            "TODATE",
            "AMOUNT",
            "PAYMENTS",
            "TRANSFERS",
            "OUTSTANDING"
        ]
    ].head(20)
)

,ID,CLAIMID,CHARGEID,PATIENTID,TYPE,FROMDATE,TODATE,AMOUNT,PAYMENTS,TRANSFERS,OUTSTANDING
43462,8bbef742-2bd9-0d95-9136-232b140553aa,0b8be9ce-dd85-8dd3-182e-32537b9d20ea,55371,0b8be9ce-dd85-8dd3-2e43-22d78bd9fa2a,CHARGE,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,61.15,0.00,NaN,0.00
43463,177df184-57b2-1b2b-226c-4956280aa755,0b8be9ce-dd85-8dd3-182e-32537b9d20ea,55372,0b8be9ce-dd85-8dd3-2e43-22d78bd9fa2a,PAYMENT,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,NaN,50.00,NaN,11.15
43464,2efbef08-af64-3656-44d8-9eac50154eaa,0b8be9ce-dd85-8dd3-182e-32537b9d20ea,55373,0b8be9ce-dd85-8dd3-2e43-22d78bd9fa2a,TRANSFEROUT,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,NaN,0.00,11.15,11.15
43465,5df7e411-5ec8-6cac-89b1-4358a02a9d54,0b8be9ce-dd85-8dd3-182e-32537b9d20ea,55374,0b8be9ce-dd85-8dd3-2e43-22d78bd9fa2a,TRANSFERIN,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,11.15,0.00,11.15,11.15
43466,bbeff822-bd90-d958-1362-b6b140553aa9,0b8be9ce-dd85-8dd3-182e-32537b9d20ea,55375,0b8be9ce-dd85-8dd3-2e43-22d78bd9fa2a,PAYMENT,1968-01-31T09:39:07Z,1970-01-01T00:00:00Z,NaN,11.15,NaN,0.00
82190,b72afad4-e955-def9-8e4e-6d81092d8fd1,f014278a-8bb2-1449-2cc6-d021ce7dc16f,106372,f014278a-8bb2-1449-c17c-e2a06048ddac,CHARGE,1968-09-14T01:20:59Z,1970-01-01T00:00:00Z,67.95,0.00,NaN,0.00
82191,6e55f5ac-d2ab-bdf3-1c9c-db05125b1fa3,f014278a-8bb2-1449-2cc6-d021ce7dc16f,106373,f014278a-8bb2-1449-c17c-e2a06048ddac,TRANSFEROUT,1968-09-14T01:20:59Z,1970-01-01T00:00:00Z,NaN,0.00,67.95,67.95
82192,dcabeb65-a557-7be6-3939-b61624b63f46,f014278a-8bb2-1449-2cc6-d021ce7dc16f,106374,f014278a-8bb2-1449-c17c-e2a06048ddac,TRANSFERIN,1968-09-14T01:20:59Z,1970-01-01T00:00:00Z,67.95,0.00,67.95,67.95
82193,b957d6d1-4aae-f7cd-7273-6c32496c7e8c,f014278a-8bb2-1449-2cc6-d021ce7dc16f,106375,f014278a-8bb2-1449-c17c-e2a06048ddac,PAYMENT,1968-09-14T01:20:59Z,1970-01-01T00:00:00Z,NaN,67.95,NaN,0.00
118744,ab4f8c81-dd85-0e1d-4872-5a130bedc5bd,68cba37b-0be0-56a0-2b5a-44812fc109e2,153252,68cba37b-0be0-56a0-7fca-313f509025cf,CHARGE,1945-05-07T03:33:59Z,1970-01-01T00:00:00Z,136.00,0.00,NaN,0.00


In [86]:
# Count the most common FROMDATE and TODATE combinations among the non-epoch violations.
non_epoch_date_pairs = (
    epoch_not_violating[
        ["FROMDATE", "TODATE"]
    ]
    .value_counts()
    .reset_index(name="count")
)

display(non_epoch_date_pairs.head(20))

,FROMDATE,TODATE,count
0,1969-03-12T22:02:16Z,1970-01-01T00:00:00Z,35
1,1966-11-01T15:56:23Z,1970-01-01T00:00:00Z,22
2,1956-01-12T07:14:06Z,1970-01-01T00:00:00Z,22
3,1967-10-25T22:02:16Z,1970-01-01T00:00:00Z,20
4,1956-07-22T05:48:05Z,1970-01-01T00:00:00Z,20
5,1967-06-30T17:44:41Z,1970-01-01T00:00:00Z,17
6,1964-01-03T17:44:41Z,1970-01-01T00:00:00Z,13
7,1958-03-19T07:59:53Z,1970-01-01T00:00:00Z,13
8,1964-02-26T00:00:44Z,1970-01-01T00:00:00Z,12
9,1930-04-20T04:24:54Z,1970-01-01T00:00:00Z,12


In [87]:
# Show which transaction types are represented among the non-epoch temporal violations.
violation_types = (
    epoch_not_violating["TYPE"]
    .value_counts()
    .reset_index(name="count")
)

display(violation_types)

,TYPE,count
0,PAYMENT,732
1,CHARGE,541
2,TRANSFEROUT,291
3,TRANSFERIN,291


In [88]:
# Classify epoch-TODATE records according to whether FROMDATE is before,
# exactly at, or after 1970-01-01 UTC.
epoch_fromdate_analysis = tables["claims_transactions"][
    tables["claims_transactions"]["TODATE"] == "1970-01-01T00:00:00Z"
].copy()

# Parse FROMDATE as UTC-aware datetime values.
epoch_fromdate_analysis["FROMDATE_PARSED"] = pd.to_datetime(
    epoch_fromdate_analysis["FROMDATE"],
    errors="coerce",
    utc=True
)

# Create a timezone-aware 1970-01-01 UTC reference point.
epoch_reference = pd.Timestamp("1970-01-01", tz="UTC")

# Classify each FROMDATE relative to the 1970-01-01 UTC reference.
epoch_fromdate_analysis["FROMDATE_PERIOD"] = np.select(
    [
        epoch_fromdate_analysis["FROMDATE_PARSED"] < epoch_reference,
        epoch_fromdate_analysis["FROMDATE_PARSED"] == epoch_reference,
        epoch_fromdate_analysis["FROMDATE_PARSED"] > epoch_reference
    ],
    [
        "Before 1970",
        "Exactly 1970",
        "After 1970"
    ],
    default="Unknown"
)

# Count records in each temporal category.
display(
    epoch_fromdate_analysis["FROMDATE_PERIOD"]
    .value_counts()
    .rename_axis("FROMDATE period")
    .reset_index(name="count")
)

,FROMDATE period,count
0,After 1970,370360
1,Before 1970,1855


In [89]:
# Isolate temporal violations where TODATE is not the suspicious 1970-01-01 value.
non_epoch_violations = claims_tx_dates[
    (from_after_to) &
    (claims_tx_dates["TODATE"] != "1970-01-01T00:00:00Z")
].copy()

print("Non-epoch temporal violations:", len(non_epoch_violations))

Non-epoch temporal violations: 1171


In [90]:
# Inspect representative non-epoch temporal violations to look for a common pattern.
display(
    non_epoch_violations[
        [
            "ID",
            "CLAIMID",
            "CHARGEID",
            "PATIENTID",
            "TYPE",
            "FROMDATE",
            "TODATE",
            "AMOUNT",
            "PAYMENTS",
            "TRANSFERS",
            "OUTSTANDING"
        ]
    ].head(20)
)

,ID,CLAIMID,CHARGEID,PATIENTID,TYPE,FROMDATE,TODATE,AMOUNT,PAYMENTS,TRANSFERS,OUTSTANDING
132953,b0f7216f-6f74-9f69-6c98-0c91b96fdd97,813075ca-b8a1-6c02-94d0-91b3ecce48b7,173006,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,CHARGE,2012-09-07T13:12:29Z,2012-09-01T13:17:19Z,507.86,0.00,NaN,0.00
132954,61ee42de-e2e9-3ed3-d930-19237edfbb2e,813075ca-b8a1-6c02-94d0-91b3ecce48b7,173007,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,PAYMENT,2012-09-07T13:12:29Z,2012-09-01T13:17:19Z,NaN,35.00,NaN,472.86
132955,c3dc85bd-dbd2-7da6-b260-324703bf765d,813075ca-b8a1-6c02-94d0-91b3ecce48b7,173008,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,PAYMENT,2012-09-07T13:12:29Z,2012-09-01T13:17:19Z,NaN,472.86,NaN,0.00
132956,1bd71016-736b-26ea-d777-fb38bd666517,813075ca-b8a1-6c02-c5df-77f8faeff69c,173010,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,CHARGE,2012-09-07T13:12:29Z,2012-09-01T13:17:19Z,1.37,0.00,NaN,0.00
132957,37ae202c-ead6-4dd4-aeef-f67186ccca2f,813075ca-b8a1-6c02-c5df-77f8faeff69c,173011,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,PAYMENT,2012-09-07T13:12:29Z,2012-09-01T13:17:19Z,NaN,1.37,NaN,0.00
134338,0bcde6e1-c188-be0f-070c-14d9172e8059,813075ca-b8a1-6c02-3b6b-dc04eae9ddc0,175071,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,CHARGE,2014-09-19T13:12:29Z,2014-09-13T13:15:43Z,155.67,0.00,NaN,0.00
134339,179bcdc3-8311-7c22-0e18-29b22e5d00be,813075ca-b8a1-6c02-3b6b-dc04eae9ddc0,175072,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,PAYMENT,2014-09-19T13:12:29Z,2014-09-13T13:15:43Z,NaN,155.67,NaN,0.00
134340,e69a5dbf-195d-1624-85b1-5969ec3e3d22,813075ca-b8a1-6c02-7d4e-3adcd09350b3,175074,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,CHARGE,2014-09-19T13:12:29Z,2014-09-13T13:15:43Z,0.91,0.00,NaN,0.00
134341,cd34bb7e-32ba-3349-0b62-b2d3d87c7d45,813075ca-b8a1-6c02-7d4e-3adcd09350b3,175075,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,PAYMENT,2014-09-19T13:12:29Z,2014-09-13T13:15:43Z,NaN,0.91,NaN,0.00
135607,469fcbee-8030-8c6a-3398-84baa9b21592,813075ca-b8a1-6c02-602e-37efe90aa533,176966,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,CHARGE,2016-09-30T13:12:29Z,2016-09-24T13:16:06Z,253.20,0.00,NaN,0.00


In [91]:
# Determine whether the non-epoch temporal violations are concentrated in particular transaction types.
violation_types = (
    non_epoch_violations["TYPE"]
    .value_counts()
    .rename_axis("TYPE")
    .reset_index(name="count")
)

display(violation_types)

,TYPE,count
0,PAYMENT,454
1,CHARGE,287
2,TRANSFEROUT,215
3,TRANSFERIN,215


In [92]:
# Identify the most common FROMDATE/TODATE combinations among the non-epoch violations.
non_epoch_date_pairs = (
    non_epoch_violations[
        ["FROMDATE", "TODATE"]
    ]
    .value_counts()
    .reset_index(name="count")
)

display(non_epoch_date_pairs.head(20))

,FROMDATE,TODATE,count
0,2008-10-21T06:31:15Z,2008-10-15T06:35:42Z,42
1,2006-10-10T06:31:15Z,2006-10-04T06:35:19Z,40
2,2017-12-20T08:23:00Z,2017-12-14T08:25:17Z,25
3,2015-12-02T08:23:00Z,2015-11-26T08:27:15Z,25
4,1981-04-29T22:02:16Z,1981-04-24T22:41:19Z,23
5,2018-02-26T15:20:25Z,2018-02-20T15:22:48Z,23
6,1985-03-17T22:37:46Z,1985-03-11T22:40:38Z,20
7,2022-12-16T18:25:34Z,2022-12-10T18:30:07Z,18
8,1979-07-20T21:46:03Z,1979-07-14T21:49:06Z,18
9,2024-12-27T18:25:34Z,2024-12-21T18:30:19Z,18


# Calculate the time difference

FROMDATE - TODATE

A positive value means the temporal ordering is reversed.

In [93]:
# Calculate exactly how far FROMDATE occurs after TODATE for the 1,171 non-epoch violations.

non_epoch_violations["FROMDATE_PARSED"] = pd.to_datetime(
    non_epoch_violations["FROMDATE"],
    utc=True,
    errors="coerce"
)

non_epoch_violations["TODATE_PARSED"] = pd.to_datetime(
    non_epoch_violations["TODATE"],
    utc=True,
    errors="coerce"
)

non_epoch_violations["difference"] = (
    non_epoch_violations["FROMDATE_PARSED"]
    - non_epoch_violations["TODATE_PARSED"]
)

display(
    non_epoch_violations["difference"]
    .describe()
)

count                         1171
mean     5 days 18:07:00.622544833
std      0 days 22:06:22.325644274
min                0 days 00:00:44
25%                5 days 23:55:22
50%                5 days 23:55:59
75%                5 days 23:57:09
max                6 days 00:00:00
Name: difference, dtype: object

In [94]:
# This will tell us the minimum, maximum, median, and quartiles of the temporal reversal.

# Convert the difference into days. This makes the pattern much easier to understand.

# Convert the temporal difference into days so we can identify recurring reversal intervals.
non_epoch_violations["difference_days"] = (
    non_epoch_violations["difference"].dt.total_seconds() / 86400
)

display(
    non_epoch_violations["difference_days"]
    .round(2)
    .describe()
)

# Find the most common reversal intervals. This is especially useful because your output already suggests something around 5–6 days.

# Count the most common temporal reversal intervals among the non-epoch violations.
difference_distribution = (
    non_epoch_violations["difference_days"]
    .round(2)
    .value_counts()
    .sort_values(ascending=False)
    .reset_index(name="count")
)

display(
    difference_distribution.head(20)
)

count    1171.000000
mean        5.757062
std         0.921596
min         0.000000
25%         6.000000
50%         6.000000
75%         6.000000
max         6.000000
Name: difference_days, dtype: float64

,difference_days,count
0,6.00,970
1,5.98,92
2,2.00,35
3,5.00,29
4,4.97,23
5,4.00,10
6,0.00,5
7,0.01,5
8,1.00,2


In [95]:
# Recalculate elapsed hours and calendar-day differences using timezone-aware timestamps.
non_epoch_violations["elapsed_hours"] = (
    non_epoch_violations["FROMDATE_PARSED"]
    - non_epoch_violations["TODATE_PARSED"]
).dt.total_seconds() / 3600

non_epoch_violations["calendar_day_difference"] = (
    non_epoch_violations["FROMDATE_PARSED"].dt.date
    .map(pd.Timestamp)
    .sub(
        non_epoch_violations["TODATE_PARSED"].dt.date.map(pd.Timestamp)
    )
    .dt.days
)

# Summarize the two measures independently.
display(
    non_epoch_violations[
        ["elapsed_hours", "calendar_day_difference"]
    ].describe()
)

,elapsed_hours,calendar_day_difference
count,1171.000000,1171.000000
mean,138.116840,5.759180
std,22.106202,0.921764
min,0.012222,0.000000
25%,143.922778,6.000000
50%,143.933056,6.000000
75%,143.952500,6.000000
max,144.000000,6.000000


In [96]:
# Compare reversal counts and elapsed-time statistics across transaction types.
display(
    non_epoch_violations.groupby("TYPE").agg(
        record_count=("ID", "size"),
        median_elapsed_hours=("elapsed_hours", "median"),
        minimum_elapsed_hours=("elapsed_hours", "min"),
        maximum_elapsed_hours=("elapsed_hours", "max")
    ).sort_values("record_count", ascending=False)
)

,record_count,median_elapsed_hours,minimum_elapsed_hours,maximum_elapsed_hours
TYPE,,,,
PAYMENT,454,143.932222,0.012222,144.0
CHARGE,287,143.937222,0.012222,144.0
TRANSFERIN,215,143.933056,0.012222,144.0
TRANSFEROUT,215,143.933056,0.012222,144.0


### Claims Transaction Temporal Consistency — Investigation Finding

The `claims_transactions` table contains 1,171 records where `FROMDATE` occurs after `TODATE`, excluding records with the suspicious `TODATE` value `1970-01-01T00:00:00Z`.

The elapsed-time distribution is highly concentrated around six days. Of the 1,171 violations, 970 (approximately 82.8%) have a difference of six days in the reported rounded distribution.

This concentration suggests a systematic temporal pattern rather than uniformly random date errors. However, the cause and business meaning of the pattern have not yet been established.

**Processing decision:** Preserve the original records and timestamps. Do not reverse the dates, apply a fixed date shift, or delete the records at this stage. Continue investigating elapsed-time differences and transaction-type distributions before deciding how to handle the violations in the processed dataset.

# Temporal Consistency Validation.

Missing end dates are not automatically errors. For example, a condition may still be active.

The code converts dates for checking only; it does not modify the original tables.

A flagged record is an anomaly to investigate, not an automatic instruction to delete it.

For patients, the comparison checks whether a recorded death date precedes the birth date.

In [98]:
# Check whether selected healthcare records have an end date earlier than their start date.
temporal_pairs = {
    "careplans": ("START", "STOP"),
    "conditions": ("START", "STOP"),
    "devices": ("START", "STOP"),
    "encounters": ("START", "STOP"),
    "medications": ("START", "STOP"),
    "procedures": ("START", "STOP"),
    "payer_transitions": ("START_DATE", "END_DATE"),
    "patients": ("BIRTHDATE", "DEATHDATE")
}

temporal_results = []

for table_name, (start_col, end_col) in temporal_pairs.items():
    df = tables[table_name]

    start = pd.to_datetime(df[start_col], errors="coerce", utc=True)
    end = pd.to_datetime(df[end_col], errors="coerce", utc=True)

    # Compare only records where both dates are available.
    comparable = start.notna() & end.notna()
    violations = comparable & (start > end)

    temporal_results.append({
        "table": table_name,
        "start_column": start_col,
        "end_column": end_col,
        "total_rows": len(df),
        "records_with_both_dates": int(comparable.sum()),
        "missing_start": int(start.isna().sum()),
        "missing_end": int(end.isna().sum()),
        "end_before_start": int(violations.sum())
    })

temporal_consistency_df = pd.DataFrame(temporal_results)

display(temporal_consistency_df)

,table,start_column,end_column,total_rows,records_with_both_dates,missing_start,missing_end,end_before_start
0,careplans,START,STOP,18976,9608,0,9368,0
1,conditions,START,STOP,208363,154571,0,53792,0
2,devices,START,STOP,33119,21087,0,12032,0
3,encounters,START,STOP,337227,337227,0,0,0
4,medications,START,STOP,292451,277446,0,15005,285
5,procedures,START,STOP,932402,932402,0,0,2
6,payer_transitions,START_DATE,END_DATE,213917,213917,0,0,0
7,patients,BIRTHDATE,DEATHDATE,5755,755,0,5000,0


In [99]:
# Identify medication records whose STOP date precedes their START date.
medication_date_violations = tables["medications"].copy()

medication_date_violations["START_PARSED"] = pd.to_datetime(
    medication_date_violations["START"],
    errors="coerce",
    utc=True
)

medication_date_violations["STOP_PARSED"] = pd.to_datetime(
    medication_date_violations["STOP"],
    errors="coerce",
    utc=True
)

medication_date_violations = medication_date_violations[
    medication_date_violations["START_PARSED"].notna()
    & medication_date_violations["STOP_PARSED"].notna()
    & (
        medication_date_violations["STOP_PARSED"]
        < medication_date_violations["START_PARSED"]
    )
]

print("Medication date violations:", len(medication_date_violations))

# Display a sample to see how far the dates are reversed.
display(
    medication_date_violations[
        [
            "PATIENT",
            "ENCOUNTER",
            "START",
            "STOP",
            "DESCRIPTION",
            "DISPENSES",
            "TOTALCOST"
        ]
    ].head(15)
)

Medication date violations: 285


,PATIENT,ENCOUNTER,START,STOP,DESCRIPTION,DISPENSES,TOTALCOST
6307,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,813075ca-b8a1-6c02-9856-6653bfab2633,2012-09-07T13:12:29Z,2012-09-01T13:17:19Z,insulin isophane human 70 UNT/ML / insulin r...,1,507.86
6308,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,813075ca-b8a1-6c02-9856-6653bfab2633,2012-09-07T13:12:29Z,2012-09-01T13:17:19Z,amLODIPine 2.5 MG Oral Tablet,1,1.37
6503,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,813075ca-b8a1-6c02-d3e0-285255a1ad41,2014-09-19T13:12:29Z,2014-09-13T13:15:43Z,insulin isophane human 70 UNT/ML / insulin r...,1,155.67
6504,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,813075ca-b8a1-6c02-d3e0-285255a1ad41,2014-09-19T13:12:29Z,2014-09-13T13:15:43Z,amLODIPine 2.5 MG Oral Tablet,1,0.91
6695,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,813075ca-b8a1-6c02-8912-3ab27e08c719,2016-09-30T13:12:29Z,2016-09-24T13:16:06Z,insulin isophane human 70 UNT/ML / insulin r...,1,253.20
6696,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,813075ca-b8a1-6c02-8912-3ab27e08c719,2016-09-30T13:12:29Z,2016-09-24T13:16:06Z,24 HR tacrolimus 1 MG Extended Release Oral Ta...,1,129.94
6697,813075ca-b8a1-6c02-e8d8-2e8b6e83af7c,813075ca-b8a1-6c02-8912-3ab27e08c719,2016-09-30T13:12:29Z,2016-09-24T13:16:06Z,amLODIPine 2.5 MG Oral Tablet,1,0.91
10169,dbe79dd1-4817-893a-6f21-b613248c1d81,dbe79dd1-4817-893a-b77a-78a2cafd343c,2017-09-15T00:26:52Z,2017-09-09T00:29:04Z,24 HR Metformin hydrochloride 500 MG Extended ...,1,2.43
10178,dbe79dd1-4817-893a-6f21-b613248c1d81,dbe79dd1-4817-893a-ea3e-853a8cb6103c,2019-09-27T00:26:52Z,2019-09-21T00:31:17Z,24 HR Metformin hydrochloride 500 MG Extended ...,1,5.12
10194,dbe79dd1-4817-893a-6f21-b613248c1d81,dbe79dd1-4817-893a-99b9-492bed369214,2021-10-08T00:26:52Z,2021-10-02T00:31:30Z,24 HR Metformin hydrochloride 500 MG Extended ...,1,1.43


In [100]:
# Display the two procedure records whose STOP dates precede their START dates.
procedure_date_violations = tables["procedures"].copy()

procedure_date_violations["START_PARSED"] = pd.to_datetime(
    procedure_date_violations["START"], errors="coerce", utc=True
)
procedure_date_violations["STOP_PARSED"] = pd.to_datetime(
    procedure_date_violations["STOP"], errors="coerce", utc=True
)

procedure_date_violations = procedure_date_violations[
    procedure_date_violations["START_PARSED"].notna()
    & procedure_date_violations["STOP_PARSED"].notna()
    & (
        procedure_date_violations["STOP_PARSED"]
        < procedure_date_violations["START_PARSED"]
    )
]

print("Procedure date violations:", len(procedure_date_violations))

display(
    procedure_date_violations[
        ["PATIENT", "ENCOUNTER", "START", "STOP", "DESCRIPTION"]
    ]
)

Procedure date violations: 2


,PATIENT,ENCOUNTER,START,STOP,DESCRIPTION
406793,1b2ff487-526c-7750-d2f8-1acf060601d8,1b2ff487-526c-7750-718d-1c3f4cf5e327,2022-03-20T01:15:41Z,2022-03-20T01:14:57Z,Transcatheter aortic valve implantation (proce...
744293,8ead6816-1aca-e40a-8417-c6738e9205a4,8ead6816-1aca-e40a-4fd0-0faf4861d68c,2017-12-04T20:12:30Z,2017-12-04T20:02:55Z,Administration of anesthesia AND/OR sedation (...


In [102]:
# Summarize the temporal consistency issues identified during validation.
temporal_quality_summary = pd.DataFrame([
    {
        "table": "claims_transactions",
        "issue": "TODATE equals 1970-01-01",
        "records_affected": 372215,
        "decision": "Flag for investigation; preserve raw values"
    },
    {
        "table": "claims_transactions",
        "issue": "FROMDATE occurs after TODATE",
        "records_affected": 371531,
        "decision": "Flag invalid date ordering; do not swap dates"
    },
    {
        "table": "medications",
        "issue": "STOP occurs before START",
        "records_affected": 285,
        "decision": "Flag in processed data; preserve raw values"
    },
    {
        "table": "procedures",
        "issue": "STOP occurs before START",
        "records_affected": 2,
        "decision": "Flag in processed data; preserve raw values"
    }
])

display(temporal_quality_summary)

,table,issue,records_affected,decision
0,claims_transactions,TODATE equals 1970-01-01,372215,Flag for investigation; preserve raw values
1,claims_transactions,FROMDATE occurs after TODATE,371531,Flag invalid date ordering; do not swap dates
2,medications,STOP occurs before START,285,Flag in processed data; preserve raw values
3,procedures,STOP occurs before START,2,Flag in processed data; preserve raw values


### Temporal Consistency — Findings and Decisions

**Objective:** Identify records with invalid or suspicious temporal relationships.

**Findings**
- All non-null values in the investigated temporal columns parsed successfully.
- No reversed intervals were found in careplans, conditions, devices, encounters, or payer transitions.
- The patients table contains no records where the death date precedes the birth date.
- The medications table contains 285 records where `STOP` precedes `START`. The inspected records show a recurring pattern of approximately six-day reversals.
- The procedures table contains two reversed intervals of less than ten minutes.
- The claims_transactions table contains 371,531 records where `FROMDATE` exceeds `TODATE`. A large proportion involve `TODATE` equal to `1970-01-01T00:00:00Z`, suggesting a possible default or sentinel value. The precise cause has not been established.

**Processing decisions**
1. Preserve the raw data unchanged.
2. Convert temporal fields to appropriate datetime types in the processed layer.
3. Retain the original date values for traceability.
4. Flag reversed intervals and suspicious transaction dates rather than automatically swapping dates or deleting records.
5. Exclude or handle flagged dates only where justified by the requirements of a specific analysis, and document the decision.

**Conclusion:** The temporal fields are syntactically valid, but several tables contain temporal inconsistencies requiring controlled handling during data preparation.

### cross-table consistency

In [103]:
# Summarize the key quality issues identified across the 18 raw tables.
quality_summary = pd.DataFrame([
    {
        "check": "Missing values",
        "finding": "Missingness varies across tables and columns",
        "action": "Assess missingness based on analytical use"
    },
    {
        "check": "Exact duplicates",
        "finding": "1,085 in observations; 598 in supplies",
        "action": "Remove confirmed redundant copies in processed data"
    },
    {
        "check": "Candidate identifiers",
        "finding": "Candidate keys identified for all tables",
        "action": "Validate key assumptions before enforcing constraints"
    },
    {
        "check": "Referential integrity",
        "finding": "All tested non-null references are valid",
        "action": "Preserve relationships between tables"
    },
    {
        "check": "Date formats",
        "finding": "All non-null values in tested date columns parsed successfully",
        "action": "Convert dates explicitly in processed data"
    },
    {
        "check": "Medication date ordering",
        "finding": "285 records have STOP before START",
        "action": "Flag anomalies; do not guess corrections"
    },
    {
        "check": "Procedure date ordering",
        "finding": "2 records have STOP before START",
        "action": "Flag anomalies; do not guess corrections"
    },
    {
        "check": "Claims transaction dates",
        "finding": "371,531 reversed date pairs; recurring 1970 TODATE",
        "action": "Investigate and flag before date-dependent analysis"
    }
])

display(quality_summary)

,check,finding,action
0,Missing values,Missingness varies across tables and columns,Assess missingness based on analytical use
1,Exact duplicates,"1,085 in observations; 598 in supplies",Remove confirmed redundant copies in processed...
2,Candidate identifiers,Candidate keys identified for all tables,Validate key assumptions before enforcing cons...
3,Referential integrity,All tested non-null references are valid,Preserve relationships between tables
4,Date formats,All non-null values in tested date columns par...,Convert dates explicitly in processed data
5,Medication date ordering,285 records have STOP before START,Flag anomalies; do not guess corrections
6,Procedure date ordering,2 records have STOP before START,Flag anomalies; do not guess corrections
7,Claims transaction dates,"371,531 reversed date pairs; recurring 1970 TO...",Investigate and flag before date-dependent ana...
